# Pipeline v8 final — BERTimbau + ADTC + Taxonomia + RAG integrado + avaliação manual + XAI

Esta versão parte da V7 e melhora o RAG sem colocar uma etapa contra a outra.

O objetivo **não** é comparar RAG semântico versus RAG discursivo como competidores.  
O objetivo é fazer os dois trabalharem juntos:

1. **BERTimbau pré-computado** fornece a predição neural.
2. **ADTC** audita a decisão e define zonas interpretativas.
3. **Taxonomia ADTC** nomeia os recursos irônicos mobilizados.
4. **RAG híbrido semântico** recupera candidatos amplos.
5. **Re-ranqueamento discursivo ADTC** reorganiza os candidatos por pertinência contextual.
6. **Avaliação manual** verifica se o contexto recuperado ajuda.
7. **XAI exploratória** acrescenta inspeção local por tokens/pesos, quando disponível.

Formulação metodológica:

> O BERTimbau decide; a ADTC audita; o RAG integrado busca contexto com recuperação semântica e reordenação discursiva.


## 0. Como usar no Kaggle

1. Ative GPU.
2. Adicione este pacote pelo **Add Data**.
3. Rode o notebook do início ao fim.

Este notebook:

- não treina;
- não retreina;
- não exige checkpoint salvo;
- carrega diretamente `neuralmind/bert-base-portuguese-cased`.

Ele continua usando as predições já geradas pelo BERTimbau clássico nos CSVs.


In [ ]:
# ============================================================
# 1. Imports e configuração geral
# ============================================================

import os
import re
import glob
import json
import math
import time
import zipfile
import shutil
import hashlib
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

import matplotlib.pyplot as plt


# ------------------------------------------------------------
# BERTimbau — predições pré-computadas, sem retreinamento
# ------------------------------------------------------------
RETREINAR_BERTIMBAU = False
BERT_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
BERT_EPOCHS = 3
MAX_LEN_BERT = 512
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42
SALVAR_MODELO_BERT = False  # troque para True se quiser salvar pesos do modelo; ocupa bastante espaço

# ------------------------------------------------------------
# IDPT integral + trava anti-vazamento
# ------------------------------------------------------------
USAR_IDPT_INTEGRAL = True
REMOVER_VAZAMENTO_EXATO_TREINO_TESTE = True
ABORTAR_SE_VAZAMENTO_RESTANTE = True
DIAGNOSTICO_QUASE_DUPLICATAS = True
REMOVER_QUASE_DUPLICATAS_DO_TREINO = False
LIMIAR_QUASE_DUPLICATA = 0.985

# ------------------------------------------------------------
# RAG híbrido v5 preservado
# ------------------------------------------------------------
USAR_RAG_NEURAL = True
USAR_RERANKER = True

EMBEDDING_MODELS = [
    "BAAI/bge-m3",
    "intfloat/multilingual-e5-base",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

TOP_K_FINAL = 5
CANDIDATE_POOL = 25
PESO_LEXICAL = 0.35
PESO_DENSO = 0.65

# Saídas.
OUT_DIR = Path("/kaggle/working/adtc_news_v7_xai_sem_retreino_bertimbau_simples") if os.path.exists("/kaggle/working") else Path("adtc_news_v7_xai_sem_retreino_bertimbau_simples")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Diretório de saída:", OUT_DIR)
print("USAR_IDPT_INTEGRAL:", USAR_IDPT_INTEGRAL)
print("RETREINAR_BERTIMBAU:", RETREINAR_BERTIMBAU)
print("BERT_EPOCHS:", BERT_EPOCHS, "(não usado nesta versão sem retreino)")
print("MAX_LEN_BERT:", MAX_LEN_BERT)
print("REMOVER_VAZAMENTO_EXATO_TREINO_TESTE:", REMOVER_VAZAMENTO_EXATO_TREINO_TESTE)
print("ABORTAR_SE_VAZAMENTO_RESTANTE:", ABORTAR_SE_VAZAMENTO_RESTANTE)
print("USAR_RAG_NEURAL:", USAR_RAG_NEURAL)
print("USAR_RERANKER:", USAR_RERANKER)


# ------------------------------------------------------------
# Configurações XAI
# ------------------------------------------------------------

EXECUTAR_XAI_NEURAL = True

# Caminho opcional para o modelo/checkpoint fine-tuned do BERTimbau.
# Exemplo: "/kaggle/input/meu-modelo-bertimbau/modelo_bertimbau"
CAMINHO_MODELO_BERT_XAI = "neuralmind/bert-base-portuguese-cased"

# Não use o modelo base como evidência na tese. Deixe False.
PERMITIR_MODELO_BASE_APENAS_DEMO = True

MAX_LEN_XAI = 256
N_AMOSTRAS_XAI_POR_TIPO = 2
LIME_NUM_SAMPLES = 500
IG_N_STEPS = 24
SHAP_MAX_AMOSTRAS = 6
TOP_TOKENS_XAI = 20


# ------------------------------------------------------------
# Configurações da taxonomia ADTC
# ------------------------------------------------------------

ARQUIVO_TAXONOMIA_ADTC = "quadro_taxonomia_ironia_pb_integrado_v7.csv"
DOCX_TAXONOMIA_ADTC = "QUADRO_TAXONOMIA_IRONIA_PB_ampliado(1).docx"


# ------------------------------------------------------------
# Configurações XAI sem retreinamento com modelo salvo
# ------------------------------------------------------------

EXECUTAR_XAI_NEURAL = True
EXIGIR_MODELO_XAI = False
CAMINHO_MODELO_BERT_XAI = "neuralmind/bert-base-portuguese-cased"
PERMITIR_MODELO_BASE_APENAS_DEMO = True

MAX_LEN_XAI = 256
N_AMOSTRAS_XAI_POR_TIPO = 2
LIME_NUM_SAMPLES = 300
IG_N_STEPS = 16
SHAP_MAX_AMOSTRAS = 4
TOP_TOKENS_XAI = 20

ARQUIVO_TAXONOMIA_ADTC = "quadro_taxonomia_ironia_pb_integrado_v7.csv"
DOCX_TAXONOMIA_ADTC = "QUADRO_TAXONOMIA_IRONIA_PB_ampliado(1).docx"


# Modo desta versão:
MODO_XAI_BERTIMBAU_SIMPLES = True


In [ ]:
# ============================================================
# 2. Preparação de arquivos no Kaggle/local
# ============================================================

def extrair_zips_kaggle(destino=OUT_DIR):
    """Extrai ZIPs encontrados em /kaggle/input para /kaggle/working/adtc_news_v6_integral."""
    if not os.path.exists("/kaggle/input"):
        return
    zips = [p for p in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(p) and p.lower().endswith(".zip")]
    for zp in zips:
        try:
            with zipfile.ZipFile(zp, "r") as z:
                z.extractall(destino)
            print("ZIP extraído:", zp)
        except Exception as e:
            print("Não foi possível extrair", zp, "->", repr(e))

extrair_zips_kaggle()


def localizar_arquivo(padroes, obrigatorio=True):
    """Localiza arquivo em Kaggle, working, subpastas e diretório local."""
    if isinstance(padroes, str):
        padroes = [padroes]
    bases = [
        ".",
        str(OUT_DIR),
        "/kaggle/working",
        "/kaggle/working/adtc_news",
        "/kaggle/working/adtc_news_v6_integral",
        "/kaggle/input",
        "/mnt/data",
    ]
    achados = []
    for base in bases:
        if not os.path.exists(base):
            continue
        for pad in padroes:
            achados.extend(glob.glob(os.path.join(base, "**", pad), recursive=True))
    achados = [a for a in achados if os.path.isfile(a)]
    achados = sorted(set(achados), key=lambda p: (len(p), p))
    if achados:
        return achados[0]
    if obrigatorio:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para: {padroes}")
    return None


def ler_csv_robusto(caminho):
    """Lê CSV/TSV de forma robusta, sem descartar linhas silenciosamente."""
    with open(caminho, "r", encoding="utf-8", errors="ignore") as f:
        primeira = f.readline()
    if "	" in primeira:
        return pd.read_csv(caminho, sep="	", engine="python", encoding="utf-8")
    if primeira.count(";") > primeira.count(","):
        return pd.read_csv(caminho, sep=";", engine="python", encoding="utf-8")
    try:
        return pd.read_csv(caminho, sep=",", engine="python", encoding="utf-8")
    except pd.errors.ParserError:
        return pd.read_csv(
            caminho,
            sep=",",
            engine="python",
            encoding="utf-8",
            quotechar='"',
            escapechar="\\",
            on_bad_lines="warn",
        )

print("Arquivos CSV encontrados:")
for f in glob.glob(str(OUT_DIR / "**" / "*.csv"), recursive=True)[:80]:
    print(f)


In [ ]:
# ============================================================
# 3. IDPT integral + trava anti-vazamento + predições BERTimbau já geradas
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold, train_test_split

# ------------------------------------------------------------
# 3.1 Normalização textual e detecção de colunas
# ------------------------------------------------------------

LABEL_HASH_BERT = re.compile(r"#(?:ironia|sarcasmo|ironic|irony|sqn)\b", re.I)
URL_BERT = re.compile(r"http\S+|www\.\S+")


def normalizar_texto_bert(s):
    """
    Normalização usada na trava anti-vazamento.
    Não altera o arquivo de predições; serve para comparar treino/teste.
    """
    if not isinstance(s, str):
        s = ""
    s = s.replace("\u00a0", " ")
    s = LABEL_HASH_BERT.sub(" ", s)
    s = URL_BERT.sub(" URL ", s)
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s


def text_hash_bert(s):
    return hashlib.md5(normalizar_texto_bert(s).encode("utf-8")).hexdigest()


def achar_coluna_texto_bert(df):
    for c in ["text", "text_limpo", "texto", "texto_limpo", "noticia", "notícia", "conteudo", "conteúdo", "content", "body"]:
        if c in df.columns:
            return c
    raise ValueError(f"Nenhuma coluna de texto encontrada. Colunas: {df.columns.tolist()}")


def achar_coluna_label_bert(df):
    for c in ["label", "rotulo", "rótulo", "classe", "target", "y", "y_true", "prediction"]:
        if c in df.columns:
            return c
    raise ValueError(f"Nenhuma coluna de rótulo encontrada. Colunas: {df.columns.tolist()}")


def normalizar_label_bert(x):
    """Converte rótulos possíveis para 0/1. 0=não irônico; 1=irônico/satírico."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return int(x)
    s = str(x).strip().strip('"').strip("'").strip().lower()
    mapa = {
        "0": 0, "1": 1,
        "false": 0, "true": 1,
        "real": 0, "not_iro": 0, "not_ironic": 0, "non-ironic": 0,
        "nao_ironico": 0, "não_irônico": 0, "nao ironico": 0, "não irônico": 0,
        "nao-irônico": 0, "não-ironico": 0, "não-irônico": 0,
        "ironic": 1, "ironico": 1, "irônico": 1,
        "satirical": 1, "satirico": 1, "satírico": 1,
        "satire": 1, "satira": 1, "sátira": 1,
    }
    if s in mapa:
        return mapa[s]
    raise ValueError(f"Rótulo não reconhecido: {x}")


def garantir_tipo_erro(df):
    if "tipo_erro" not in df.columns and {"label", "bert_pred"}.issubset(df.columns):
        def tipo_erro(row):
            y = int(row["label"])
            p = int(row["bert_pred"])
            if y == 0 and p == 0: return "VN"
            if y == 0 and p == 1: return "FP"
            if y == 1 and p == 0: return "FN"
            if y == 1 and p == 1: return "VP"
            return "NA"
        df["tipo_erro"] = df.apply(tipo_erro, axis=1)
    return df


# ------------------------------------------------------------
# 3.2 Carregamento integral do IDPT News
# ------------------------------------------------------------

arquivo_treino_raw = localizar_arquivo([
    "training_news.csv",
    "training_news*.csv",
])

arquivo_teste_raw = localizar_arquivo([
    "test_news_com_rotulo.csv",
    "test_news_com_rotulo*.csv",
])

train_raw_original = ler_csv_robusto(arquivo_treino_raw)
test_raw_original = ler_csv_robusto(arquivo_teste_raw)

print("Arquivo treino integral:", arquivo_treino_raw)
print("Arquivo teste:", arquivo_teste_raw)
print("Shape treino bruto:", train_raw_original.shape)
print("Shape teste bruto:", test_raw_original.shape)
print("Colunas treino:", train_raw_original.columns.tolist())
print("Colunas teste:", test_raw_original.columns.tolist())

col_text_train = achar_coluna_texto_bert(train_raw_original)
col_label_train = achar_coluna_label_bert(train_raw_original)
col_text_test = achar_coluna_texto_bert(test_raw_original)
col_label_test = achar_coluna_label_bert(test_raw_original)

train_raw = train_raw_original.copy()
test_raw = test_raw_original.copy()

train_raw["texto_modelo"] = train_raw[col_text_train].map(normalizar_texto_bert)
test_raw["texto_modelo"] = test_raw[col_text_test].map(normalizar_texto_bert)
train_raw["label"] = train_raw[col_label_train].apply(normalizar_label_bert).astype(int)
test_raw["label"] = test_raw[col_label_test].apply(normalizar_label_bert).astype(int)
train_raw["text_hash"] = train_raw["texto_modelo"].map(text_hash_bert)
test_raw["text_hash"] = test_raw["texto_modelo"].map(text_hash_bert)

if "id_texto_original" not in test_raw.columns:
    test_raw["id_texto_original"] = np.arange(len(test_raw))

print("Distribuição treino:")
display(train_raw["label"].value_counts().sort_index().to_frame("n"))
print("Distribuição teste:")
display(test_raw["label"].value_counts().sort_index().to_frame("n"))


# ------------------------------------------------------------
# 3.3 Trava anti-vazamento treino/teste
# ------------------------------------------------------------

n_treino_antes_trava = len(train_raw)
hashes_teste = set(test_raw["text_hash"])
mask_vazamento = train_raw["text_hash"].isin(hashes_teste)

linhas_vazadas = train_raw.loc[mask_vazamento].copy()
n_vazamento_exato = int(mask_vazamento.sum())

if REMOVER_VAZAMENTO_EXATO_TREINO_TESTE:
    train_pos_trava = train_raw.loc[~mask_vazamento].copy()
else:
    train_pos_trava = train_raw.copy()

n_treino_depois_trava = len(train_pos_trava)
n_overlap_pos = len(set(train_pos_trava["text_hash"]) & set(test_raw["text_hash"]))

diagnostico_antivazamento = pd.DataFrame({
    "item": [
        "treino_bruto_original",
        "teste",
        "vazamentos_exatos_treino_teste_encontrados",
        "linhas_removidas_do_treino",
        "treino_pos_trava",
        "overlap_hash_pos_trava",
    ],
    "valor": [
        n_treino_antes_trava,
        len(test_raw),
        n_vazamento_exato,
        n_vazamento_exato if REMOVER_VAZAMENTO_EXATO_TREINO_TESTE else 0,
        n_treino_depois_trava,
        n_overlap_pos,
    ]
})

diagnostico_antivazamento.to_csv(
    OUT_DIR / "diagnostico_antivazamento_treino_teste_v6_sem_retreino.csv",
    index=False
)

linhas_vazadas.to_csv(
    OUT_DIR / "linhas_treino_removidas_por_vazamento_teste_v6_sem_retreino.csv",
    index=False
)

display(diagnostico_antivazamento)

if ABORTAR_SE_VAZAMENTO_RESTANTE and n_overlap_pos > 0:
    raise RuntimeError(
        f"Vazamento treino/teste ainda presente após trava: {n_overlap_pos} hashes em comum."
    )


# ------------------------------------------------------------
# 3.4 Diagnóstico de split treino/validação por grupo de hash
# ------------------------------------------------------------

groups = train_pos_trava["text_hash"].values
y_all = train_pos_trava["label"].values

try:
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_idx, val_idx = next(sgkf.split(train_pos_trava, y_all, groups=groups))
except Exception as e:
    warnings.warn(f"StratifiedGroupKFold falhou ({repr(e)}). Usando train_test_split estratificado.")
    tr_idx, val_idx = train_test_split(
        np.arange(len(train_pos_trava)),
        test_size=0.10,
        random_state=SEED,
        stratify=y_all,
    )

tr_diag = train_pos_trava.iloc[tr_idx].reset_index(drop=True)
val_diag = train_pos_trava.iloc[val_idx].reset_index(drop=True)

n_overlap_train_val = len(set(tr_diag["text_hash"]) & set(val_diag["text_hash"]))

diagnostico_split = pd.DataFrame({
    "item": [
        "treino_diagnostico",
        "validacao_diagnostico",
        "hashes_comuns_treino_validacao",
    ],
    "valor": [
        len(tr_diag),
        len(val_diag),
        n_overlap_train_val,
    ]
})

diagnostico_split.to_csv(
    OUT_DIR / "diagnostico_split_train_val_v6_sem_retreino.csv",
    index=False
)

display(diagnostico_split)

if ABORTAR_SE_VAZAMENTO_RESTANTE and n_overlap_train_val > 0:
    raise RuntimeError(
        f"Vazamento treino/validação no diagnóstico: {n_overlap_train_val} hashes em comum."
    )


# ------------------------------------------------------------
# 3.5 Diagnóstico opcional de quase duplicatas treino/teste
# ------------------------------------------------------------

if DIAGNOSTICO_QUASE_DUPLICATAS:
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity

        amostra_treino = train_pos_trava[["text_hash", "texto_modelo"]].drop_duplicates("text_hash").copy()
        amostra_teste = test_raw[["text_hash", "texto_modelo"]].drop_duplicates("text_hash").copy()

        corpus_diag = pd.concat([amostra_treino["texto_modelo"], amostra_teste["texto_modelo"]], ignore_index=True)
        vec_diag = TfidfVectorizer(max_features=50000, ngram_range=(3, 5), analyzer="char")
        X_diag = vec_diag.fit_transform(corpus_diag)

        X_tr = X_diag[:len(amostra_treino)]
        X_te = X_diag[len(amostra_treino):]

        sim = cosine_similarity(X_te, X_tr)
        max_sim = sim.max(axis=1)
        argmax_sim = sim.argmax(axis=1)

        quase = pd.DataFrame({
            "test_hash": amostra_teste["text_hash"].values,
            "train_hash_mais_proximo": amostra_treino["text_hash"].values[argmax_sim],
            "similaridade_char_tfidf": max_sim,
        })

        quase = quase[quase["similaridade_char_tfidf"] >= LIMIAR_QUASE_DUPLICATA].copy()
        quase.to_csv(
            OUT_DIR / "candidatos_quase_duplicados_treino_teste_v6_sem_retreino.csv",
            index=False
        )
        print("Quase duplicatas acima do limiar:", len(quase))

    except Exception as e:
        warnings.warn(f"Diagnóstico de quase duplicatas não executado: {repr(e)}")
        quase = pd.DataFrame()
else:
    quase = pd.DataFrame()


# ------------------------------------------------------------
# 3.6 Carregar predições já geradas pelo BERTimbau
# ------------------------------------------------------------

arquivo_pred_teste = localizar_arquivo([
    "idpt_news_predicoes_teste_bertimbau_classico.csv",
    "idpt_news_predicoes_teste*.csv",
])

arquivo_pred_val = localizar_arquivo([
    "idpt_news_predicoes_validacao_bertimbau_classico.csv",
    "idpt_news_predicoes_validacao*.csv",
], obrigatorio=False)

arquivo_metricas_bert = localizar_arquivo([
    "metricas_bertimbau_classico_idpt.csv",
    "metricas_bert*.csv",
], obrigatorio=False)

arquivo_matriz_bert = localizar_arquivo([
    "matriz_confusao_bertimbau_classico_idpt.csv",
    "matriz_confusao_bert*.csv",
], obrigatorio=False)

test_pred_df = ler_csv_robusto(arquivo_pred_teste)
if arquivo_pred_val is not None:
    val_pred_df = ler_csv_robusto(arquivo_pred_val)
else:
    val_pred_df = pd.DataFrame()

print("Predições de teste carregadas:", arquivo_pred_teste)
print("Predições de validação carregadas:", arquivo_pred_val)
print("Shape predições teste:", test_pred_df.shape)

# Garantir id_texto_original
if "id_texto_original" not in test_pred_df.columns:
    test_pred_df["id_texto_original"] = np.arange(len(test_pred_df))

# Garantir label, se ausente
if "label" not in test_pred_df.columns:
    if len(test_pred_df) == len(test_raw):
        test_pred_df["label"] = test_raw["label"].values
    else:
        raise ValueError("Predições de teste sem label e tamanho diferente do teste bruto.")

# Garantir bert_pred
if "bert_pred" not in test_pred_df.columns:
    candidatos_pred = ["pred", "prediction_modelo", "predicao", "predicted_label", "y_pred"]
    achou_pred = False
    for c in candidatos_pred:
        if c in test_pred_df.columns:
            test_pred_df["bert_pred"] = test_pred_df[c]
            achou_pred = True
            break
    if not achou_pred:
        raise ValueError(f"Arquivo de predições sem coluna bert_pred. Colunas: {test_pred_df.columns.tolist()}")

# Garantir texto/hash para alinhamento
if "text_hash" not in test_pred_df.columns:
    if "text" in test_pred_df.columns:
        test_pred_df["text_hash"] = test_pred_df["text"].map(text_hash_bert)
    elif len(test_pred_df) == len(test_raw):
        test_pred_df["text_hash"] = test_raw["text_hash"].values

# Garantir confiança/probabilidade
if "bert_conf" not in test_pred_df.columns:
    test_pred_df["bert_conf"] = np.nan
if "bert_p_iro" not in test_pred_df.columns:
    test_pred_df["bert_p_iro"] = np.nan

test_pred_df["label"] = test_pred_df["label"].astype(int)
test_pred_df["bert_pred"] = test_pred_df["bert_pred"].astype(int)
test_pred_df = garantir_tipo_erro(test_pred_df)

# Verificação de alinhamento com teste bruto
if len(test_pred_df) != len(test_raw):
    raise ValueError(
        f"O arquivo de predições tem {len(test_pred_df)} linhas, mas o teste tem {len(test_raw)}."
    )

if "text_hash" in test_pred_df.columns:
    alinhamento_hash = (test_pred_df["text_hash"].astype(str).values == test_raw["text_hash"].astype(str).values).mean()
else:
    alinhamento_hash = np.nan

diagnostico_predicoes = pd.DataFrame({
    "item": [
        "arquivo_predicoes_teste",
        "arquivo_predicoes_validacao",
        "linhas_predicoes_teste",
        "linhas_teste_bruto",
        "alinhamento_hash_por_ordem",
        "usa_predicoes_precomputadas",
        "bertimbau_retreinado_no_notebook",
    ],
    "valor": [
        arquivo_pred_teste,
        arquivo_pred_val if arquivo_pred_val is not None else "NA",
        len(test_pred_df),
        len(test_raw),
        alinhamento_hash,
        True,
        False,
    ]
})

diagnostico_predicoes.to_csv(
    OUT_DIR / "diagnostico_predicoes_bertimbau_precomputadas_v6_sem_retreino.csv",
    index=False
)

display(diagnostico_predicoes)

# Salvar cópias padronizadas das predições usadas
pred_teste_path = OUT_DIR / "idpt_news_predicoes_teste_precomputado_v6_sem_retreino.csv"
pred_val_path = OUT_DIR / "idpt_news_predicoes_validacao_precomputado_v6_sem_retreino.csv"

test_pred_df.to_csv(pred_teste_path, index=False)
if not val_pred_df.empty:
    val_pred_df.to_csv(pred_val_path, index=False)

# Carregar métricas/matriz se existirem, ou recalcular a partir das predições
if arquivo_metricas_bert is not None:
    metricas_bert_precomputado = ler_csv_robusto(arquivo_metricas_bert)
    metricas_bert_precomputado.to_csv(OUT_DIR / "metricas_bertimbau_precomputado_v6_sem_retreino.csv", index=False)
else:
    metricas_bert_precomputado = pd.DataFrame()

if arquivo_matriz_bert is not None:
    matriz_bert_precomputada = ler_csv_robusto(arquivo_matriz_bert)
    matriz_bert_precomputada.to_csv(OUT_DIR / "matriz_confusao_bertimbau_precomputado_v6_sem_retreino.csv", index=False)
else:
    matriz_bert_precomputada = pd.DataFrame()

print("Predições disponíveis para integração:", test_pred_df.shape)
display(test_pred_df[["id_texto_original", "label", "bert_pred", "bert_conf", "bert_p_iro", "tipo_erro"]].head())


# 4. Quadro de referência da Taxonomia ADTC

Esta seção carrega o **Quadro-Síntese da Taxonomia dos Recursos Irônicos no Português Brasileiro**.

O quadro é parte metodológica do notebook. Ele orienta a leitura das colunas ADTC, isto é, permite explicar quais recursos discursivos justificam a classificação de um texto como ironia marcada, baixa marcação contextual, falso gatilho ou não ironia.

A taxonomia contém **77 recursos**, organizados em **5 categorias**:

1. Marcadores de sinalização irônica;
2. Estruturas sintáticas;
3. Estruturas lexicais;
4. Estruturas semânticas;
5. Contexto como constituinte da ironia.

As categorias e os recursos não são mutuamente excludentes: uma mesma ocorrência pode mobilizar mais de um recurso.


In [ ]:
# ============================================================
# 4. Carregar quadro da taxonomia ADTC
# ============================================================

arquivo_taxonomia = localizar_arquivo([
    ARQUIVO_TAXONOMIA_ADTC,
    "quadro_taxonomia_ironia_pb*.csv",
    "taxonomia_ironia*.csv",
], obrigatorio=False)

if arquivo_taxonomia is None:
    raise FileNotFoundError(
        "Quadro da taxonomia ADTC não encontrado. "
        "Inclua quadro_taxonomia_ironia_pb_integrado.csv no pacote."
    )

quadro_taxonomia = ler_csv_robusto(arquivo_taxonomia)

# Padroniza nomes esperados.
quadro_taxonomia.columns = [str(c).strip() for c in quadro_taxonomia.columns]

colunas_taxonomia_esperadas = [
    "tabela", "categoria_num", "categoria", "categoria_descricao",
    "n_recurso", "recurso", "definicao_fundamentacao",
    "exemplos_pb", "features_adt", "detectabilidade",
    "observacoes_anotacao"
]

faltando = [c for c in colunas_taxonomia_esperadas if c not in quadro_taxonomia.columns]
if faltando:
    raise ValueError(f"Colunas ausentes no quadro da taxonomia: {faltando}")

quadro_taxonomia["n_recurso"] = quadro_taxonomia["n_recurso"].astype(int)
quadro_taxonomia["categoria_num"] = quadro_taxonomia["categoria_num"].astype(int)

taxonomia_resumo_categorias = (
    quadro_taxonomia
    .groupby(["categoria_num", "categoria"], dropna=False)
    .size()
    .reset_index(name="n_recursos")
    .sort_values("categoria_num")
)

taxonomia_resumo_detectabilidade = (
    quadro_taxonomia["detectabilidade"]
    .value_counts(dropna=False)
    .reset_index()
)
taxonomia_resumo_detectabilidade.columns = ["detectabilidade", "n_recursos"]

# Mapeamento entre os marcadores computacionais já extraídos pela ADTC
# e os recursos do quadro teórico.
MAPA_MARCADOR_TAXONOMIA = {
    "aspas_ironicas": [5],
    "adversativa": [27],
    "negacao_polemica": [22],
    "discurso_relatado": [35],
    "avaliativo_invertido": [37],
    "aumentativo_ironico": [39],
    "diminutivo_ironico": [38],
    "antifrase_hiperbole": [1, 2],
    "maiusculas_enf": [7],
    "contraste_polaridade": [55],
    "onomatopeia": [15],
    "interrogacao_retorica": [24],
    "pontuacao_expressiva": [6],
    "exclamacao_inversao": [25],
    "formulaicas_modal": [11, 45],
    "alongamento_vocalico": [16],
    "mencao_contexto": [64, 65, 66],
    "pretericao": [34],
    "reduplicacao": [36],
}

mapa_marcador_taxonomia_df = pd.DataFrame([
    {"marcador_adtc": marcador, "n_recurso": rid}
    for marcador, ids in MAPA_MARCADOR_TAXONOMIA.items()
    for rid in ids
]).merge(
    quadro_taxonomia[
        ["n_recurso", "recurso", "categoria_num", "categoria", "features_adt", "detectabilidade"]
    ],
    on="n_recurso",
    how="left"
)

def parse_marcadores_adtc(marcadores):
    """
    Converte a coluna de marcadores ADTC em lista de códigos.

    Exemplo:
    'aspas_ironicas:Cat1; discurso_relatado:Cat2'
    -> ['aspas_ironicas', 'discurso_relatado']
    """
    if pd.isna(marcadores):
        return []
    saida = []
    for parte in str(marcadores).split(";"):
        parte = parte.strip()
        if not parte:
            continue
        codigo = parte.split(":")[0].strip()
        saida.append(codigo)
    return saida

def recursos_taxonomicos_para_marcadores(marcadores):
    codigos = parse_marcadores_adtc(marcadores)
    ids = []
    for cod in codigos:
        ids.extend(MAPA_MARCADOR_TAXONOMIA.get(cod, []))
    ids = sorted(set(ids))
    return ids

def explicar_taxonomia_marcadores(marcadores):
    ids = recursos_taxonomicos_para_marcadores(marcadores)
    if not ids:
        return "Sem recurso taxonômico mapeado a partir dos marcadores superficiais."
    sub = quadro_taxonomia[quadro_taxonomia["n_recurso"].isin(ids)].copy()
    partes = []
    for _, r in sub.sort_values(["categoria_num", "n_recurso"]).iterrows():
        partes.append(
            f"{int(r['n_recurso'])}. {r['recurso']} "
            f"[Cat. {int(r['categoria_num'])}: {r['categoria']}; "
            f"detectabilidade: {r['detectabilidade']}]"
        )
    return " | ".join(partes)

def categorias_taxonomicas_marcadores(marcadores):
    ids = recursos_taxonomicos_para_marcadores(marcadores)
    if not ids:
        return ""
    sub = quadro_taxonomia[quadro_taxonomia["n_recurso"].isin(ids)]
    cats = [
        f"Cat. {int(r.categoria_num)} — {r.categoria}"
        for r in sub[["categoria_num", "categoria"]].drop_duplicates().itertuples(index=False)
    ]
    return " | ".join(cats)

print("Taxonomia carregada:", arquivo_taxonomia)
print("Total de recursos:", len(quadro_taxonomia))
print("Total de categorias:", quadro_taxonomia["categoria_num"].nunique())

display(taxonomia_resumo_categorias)
display(taxonomia_resumo_detectabilidade)
display(mapa_marcador_taxonomia_df.head(30))

# Salva cópias dentro do diretório de saída.
quadro_taxonomia.to_csv(OUT_DIR / "quadro_taxonomia_ironia_pb_integrado.csv", index=False)
mapa_marcador_taxonomia_df.to_csv(OUT_DIR / "mapa_marcadores_adtc_taxonomia.csv", index=False)
taxonomia_resumo_categorias.to_csv(OUT_DIR / "taxonomia_resumo_categorias.csv", index=False)
taxonomia_resumo_detectabilidade.to_csv(OUT_DIR / "taxonomia_resumo_detectabilidade.csv", index=False)


In [ ]:
# ============================================================
# 5. Carregar ADTC, base RAG e integrar predições pré-computadas
# ============================================================

arquivo_base_rag = localizar_arquivo([
    "base_news_rag_curada_v2_com_proveniencia.csv",
    "base_news_rag_curada*.csv",
])
base_rag = ler_csv_robusto(arquivo_base_rag)

# Preferir o teste com marcadores ADTC já extraídos.
arquivo_teste_adtc = localizar_arquivo([
    "idpt_news_teste_adtc_corrigido.csv",
    "idpt_news_teste_adtc_corrigido*.csv",
    # Correção pós-revisão: fallback para o teste v6 já anotado, presente no
    # pacote. Sem ele, o pipeline cairia no teste bruto com n_marc_sup=0 e
    # marcadores vazios, amputando silenciosamente a camada discursiva.
    "idpt_news_teste_com_taxonomia_adtc_v6_xai.csv",
    "idpt_news_teste_com_taxonomia_adtc*.csv",
], obrigatorio=False)

if arquivo_teste_adtc is not None:
    test = ler_csv_robusto(arquivo_teste_adtc)
    print("Teste ADTC carregado:", arquivo_teste_adtc)
else:
    print("Teste ADTC não encontrado. Usando teste bruto; n_marc_sup será definido como 0.")
    test = test_raw.copy()
    test["n_marc_sup"] = 0

# Normalizar ids e texto.
if "id_texto_original" not in test.columns:
    test["id_texto_original"] = np.arange(len(test))

# Se a coluna label do teste ADTC estiver com nome diferente, padronizar.
if "label" not in test.columns:
    for c in ["rotulo", "y", "classe", "target"]:
        if c in test.columns:
            test["label"] = test[c].astype(int)
            break
if "label" not in test.columns:
    test["label"] = test_raw["label"].values if len(test) == len(test_raw) else np.nan

# Preparar predições pré-computadas.
pred_retreino = test_pred_df.copy()
if "id_texto_original" not in pred_retreino.columns:
    pred_retreino["id_texto_original"] = np.arange(len(pred_retreino))

# Garantir colunas de interesse.
for c in ["bert_pred", "bert_conf", "bert_p_iro", "tipo_erro"]:
    if c not in pred_retreino.columns:
        if c == "bert_conf": pred_retreino[c] = np.nan
        elif c == "bert_p_iro": pred_retreino[c] = np.nan
        elif c == "tipo_erro": pred_retreino[c] = "NA"
        else: raise ValueError("Predições sem bert_pred.")

cols_pred = ["id_texto_original", "bert_pred", "bert_conf", "bert_p_iro", "tipo_erro"]
pred_min = pred_retreino[cols_pred].drop_duplicates(subset=["id_texto_original"])

# Remover predições antigas e integrar as novas.
test = test.drop(columns=[c for c in ["bert_pred", "bert_conf", "bert_p_iro", "tipo_erro"] if c in test.columns], errors="ignore")
test = test.merge(pred_min, on="id_texto_original", how="left")

# Fallback por ordem se algum id não casar.
faltantes = test["bert_pred"].isna().sum()
if faltantes > 0:
    warnings.warn(f"{faltantes} textos sem predição por id_texto_original. Tentando fallback por ordem.")
    if len(test) == len(pred_retreino):
        test["bert_pred"] = pred_retreino["bert_pred"].values
        test["bert_conf"] = pred_retreino["bert_conf"].values
        test["bert_p_iro"] = pred_retreino["bert_p_iro"].values
        test["tipo_erro"] = pred_retreino["tipo_erro"].values
    else:
        raise ValueError("Não foi possível integrar predições: tamanhos diferentes.")

test["bert_pred"] = test["bert_pred"].astype("Int64")
if "n_marc_sup" not in test.columns:
    test["n_marc_sup"] = 0

print("Base RAG:", arquivo_base_rag)
print("Shape teste integrado com predições pré-computadas:", test.shape)
print("Shape base RAG:", base_rag.shape)
print("Predições ausentes:", test["bert_pred"].isna().sum())
print("Colunas teste:", test.columns.tolist()[:80])
print("Colunas base:", base_rag.columns.tolist())

display(test.head(2))
display(base_rag.head(2))


In [ ]:
# ============================================================
# 5. Integrar taxonomia ao teste ADTC
# ============================================================

def gerar_explicacao_classificacao_taxonomica(row):
    pred = row.get("bert_pred", pd.NA)
    conf = row.get("bert_conf", np.nan)
    n_marc = row.get("n_marc_sup", 0)
    zona = row.get("zona_adtc", row.get("tipo_adtc", "zona não definida"))
    marcadores = row.get("marcadores", "")

    try:
        pred_int = int(pred)
        pred_txt = "irônico" if pred_int == 1 else "não irônico"
    except Exception:
        pred_txt = "sem predição"

    if pd.notna(conf):
        bert_txt = f"BERTimbau clássico predisse {pred_txt} com confiança {float(conf):.3f}."
    else:
        bert_txt = f"BERTimbau clássico predisse {pred_txt}."

    ids = recursos_taxonomicos_para_marcadores(marcadores)
    if ids:
        recursos_txt = explicar_taxonomia_marcadores(marcadores)
        adtc_txt = (
            f"A ADTC encontrou {n_marc} marcador(es) superficial(is), "
            f"mapeados para os seguintes recursos da taxonomia: {recursos_txt}."
        )
    else:
        adtc_txt = (
            f"A ADTC encontrou {n_marc} marcador(es) superficial(is), "
            "mas nenhum recurso taxonômico foi mapeado automaticamente."
        )

    return f"{bert_txt} Zona ADTC: {zona}. {adtc_txt}"

if "marcadores" not in test.columns:
    test["marcadores"] = ""

test["taxonomia_recurso_ids"] = test["marcadores"].apply(
    lambda x: ";".join(map(str, recursos_taxonomicos_para_marcadores(x)))
)

test["taxonomia_recursos"] = test["marcadores"].apply(explicar_taxonomia_marcadores)
test["taxonomia_categorias"] = test["marcadores"].apply(categorias_taxonomicas_marcadores)
test["explicacao_classificacao_taxonomica"] = test.apply(
    gerar_explicacao_classificacao_taxonomica,
    axis=1
)

# Tabela de evidências por texto e por recurso.
linhas_evidencias = []
for _, row in test.iterrows():
    ids = recursos_taxonomicos_para_marcadores(row.get("marcadores", ""))
    for rid in ids:
        rec = quadro_taxonomia[quadro_taxonomia["n_recurso"] == rid]
        if rec.empty:
            continue
        rec = rec.iloc[0]
        linhas_evidencias.append({
            "id_texto_original": row.get("id_texto_original", np.nan),
            "label": row.get("label", np.nan),
            "bert_pred": row.get("bert_pred", np.nan),
            "tipo_erro": row.get("tipo_erro", np.nan),
            "zona_adtc": row.get("zona_adtc", row.get("tipo_adtc", "")),
            "marcadores": row.get("marcadores", ""),
            "n_recurso": rid,
            "recurso": rec["recurso"],
            "categoria_num": rec["categoria_num"],
            "categoria": rec["categoria"],
            "features_adt": rec["features_adt"],
            "detectabilidade": rec["detectabilidade"],
        })

evidencias_taxonomia_textos = pd.DataFrame(linhas_evidencias)

test.to_csv(OUT_DIR / "idpt_news_teste_com_taxonomia_adtc_v6_xai.csv", index=False)
evidencias_taxonomia_textos.to_csv(OUT_DIR / "evidencias_taxonomia_por_texto_v6_xai.csv", index=False)

print("Taxonomia integrada ao teste.")
print("Colunas adicionadas:")
print([
    "taxonomia_recurso_ids",
    "taxonomia_recursos",
    "taxonomia_categorias",
    "explicacao_classificacao_taxonomica",
])

display(test[[
    c for c in [
        "id_texto_original", "label", "bert_pred", "tipo_erro",
        "n_marc_sup", "marcadores", "taxonomia_recurso_ids",
        "taxonomia_categorias", "explicacao_classificacao_taxonomica"
    ] if c in test.columns
]].head(5))

display(evidencias_taxonomia_textos.head(20))


In [ ]:
# ============================================================
# 6. Normalização, diagnóstico e métricas BERT/ADTC já existentes
# ============================================================

LABEL_HASH = re.compile(r"#(?:ironia|sarcasmo|ironic|irony|sqn)\b", re.I)
URL = re.compile(r"http\S+|www\.\S+")


def normalizar_texto(s):
    if not isinstance(s, str):
        s = ""
    s = LABEL_HASH.sub(" ", s)
    s = URL.sub(" URL ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def achar_coluna_texto(df):
    for c in ["text", "text_limpo", "texto", "texto_limpo", "noticia", "conteudo", "content"]:
        if c in df.columns:
            return c
    raise ValueError("Nenhuma coluna de texto encontrada.")

col_texto = achar_coluna_texto(test)
test = test.copy()
test["texto_busca_v5"] = test[col_texto].fillna("").map(normalizar_texto)

if "id_texto_original" not in test.columns:
    test["id_texto_original"] = np.arange(len(test))
if "label" in test.columns:
    test["label"] = test["label"].astype(int)
if "bert_pred" in test.columns:
    test["bert_pred"] = test["bert_pred"].astype("Int64")

# tipo_erro se ausente
if "tipo_erro" not in test.columns and {"label", "bert_pred"}.issubset(test.columns):
    def tipo_erro(row):
        y, p = int(row["label"]), int(row["bert_pred"])
        if y == 0 and p == 0: return "VN"
        if y == 0 and p == 1: return "FP"
        if y == 1 and p == 0: return "FN"
        if y == 1 and p == 1: return "VP"
        return "NA"
    test["tipo_erro"] = test.apply(tipo_erro, axis=1)

# zona ADTC se ausente
if "zona_adtc" not in test.columns:
    if "n_marc_sup" not in test.columns:
        test["n_marc_sup"] = 0
    def zona(row):
        pred = int(row["bert_pred"]) if pd.notna(row.get("bert_pred", np.nan)) else -1
        n = row.get("n_marc_sup", 0)
        if pred == 1 and n >= 3: return "Ironia predita com alta/média marcação superficial"
        if pred == 1 and n <= 2: return "Ironia predita com baixa marcação: exige contexto"
        if pred == 0 and n >= 1: return "Não ironia predita com marcadores: possível falso gatilho"
        return "Não ironia predita sem marcadores relevantes"
    test["zona_adtc"] = test.apply(zona, axis=1)

# Métricas BERT
if {"label", "bert_pred"}.issubset(test.columns) and test["bert_pred"].notna().any():
    ok = test.dropna(subset=["bert_pred"]).copy()
    ok["bert_pred"] = ok["bert_pred"].astype(int)
    y = ok["label"].astype(int)
    p = ok["bert_pred"].astype(int)
    cm = confusion_matrix(y, p, labels=[0, 1])
    metricas_bert = pd.DataFrame({
        "metrica": ["accuracy", "balanced_accuracy", "macro_f1"],
        "valor": [accuracy_score(y, p), balanced_accuracy_score(y, p), f1_score(y, p, average="macro")]
    })
    print("Matriz de confusão [[VN, FP], [FN, VP]]:")
    print(cm)
    display(metricas_bert)
else:
    cm = np.array([[np.nan, np.nan], [np.nan, np.nan]])
    metricas_bert = pd.DataFrame({"metrica": [], "valor": []})
    print("Sem label/bert_pred para métricas BERT.")

print("Total teste:", len(test))
print("Zonas ADTC:")
display(test["zona_adtc"].value_counts(dropna=False).to_frame("n"))


In [ ]:
# ============================================================
# 7. Motor RAG v5: híbrido lexical + embeddings + reranker
# ============================================================


def normalizar_score(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return x
    mn, mx = np.nanmin(x), np.nanmax(x)
    if not np.isfinite(mn) or not np.isfinite(mx) or mx - mn < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - mn) / (mx - mn)


class MotorRAGHibridoV5:
    def __init__(
        self,
        base_df,
        embedding_models=EMBEDDING_MODELS,
        reranker_model=RERANKER_MODEL,
        usar_neural=USAR_RAG_NEURAL,
        usar_reranker=USAR_RERANKER,
        peso_lexical=PESO_LEXICAL,
        peso_denso=PESO_DENSO,
        candidate_pool=CANDIDATE_POOL,
    ):
        self.base = base_df.copy().reset_index(drop=True)
        for c in ["id", "tema", "fragmento", "fonte_sugerida"]:
            if c not in self.base.columns:
                self.base[c] = ""
        self.docs = (
            self.base["tema"].fillna("").astype(str) + " | " +
            self.base["fragmento"].fillna("").astype(str)
        ).map(normalizar_texto).tolist()
        self.ids = self.base["id"].astype(str).tolist()
        self.temas = self.base["tema"].astype(str).tolist()
        self.fragmentos = self.base["fragmento"].astype(str).tolist()
        self.fontes = self.base["fonte_sugerida"].astype(str).tolist()
        self.peso_lexical = peso_lexical
        self.peso_denso = peso_denso
        self.candidate_pool = candidate_pool
        self.embedding_models = embedding_models
        self.reranker_model = reranker_model
        self.usar_neural = usar_neural
        self.usar_reranker = usar_reranker
        self.motor = "tfidf_fallback"

        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True,
        )
        self.X_lex = self.vectorizer.fit_transform(self.docs)

        self.embedder = None
        self.doc_embeddings = None
        self.reranker = None
        self.modelo_embedding_usado = None
        self.modelo_reranker_usado = None

        if self.usar_neural:
            self._tentar_carregar_embedder()
            self._tentar_carregar_reranker()

    def _tentar_carregar_embedder(self):
        try:
            from sentence_transformers import SentenceTransformer
        except Exception as e:
            print("sentence-transformers indisponível. Usando TF-IDF. Motivo:", repr(e))
            return
        for model_name in self.embedding_models:
            try:
                print("Tentando carregar embedder:", model_name)
                self.embedder = SentenceTransformer(model_name)
                self.doc_embeddings = self.embedder.encode(
                    self.docs,
                    normalize_embeddings=True,
                    show_progress_bar=False,
                    batch_size=16,
                )
                self.modelo_embedding_usado = model_name
                self.motor = "hybrid_bge" if "bge" in model_name.lower() else "hybrid_dense"
                print("Embedder carregado:", model_name)
                return
            except Exception as e:
                print("Falha ao carregar embedder", model_name, "->", repr(e))
        self.embedder = None
        self.doc_embeddings = None
        self.motor = "tfidf_fallback"

    def _tentar_carregar_reranker(self):
        if self.embedder is None or not self.usar_reranker:
            return
        try:
            from sentence_transformers import CrossEncoder
            print("Tentando carregar reranker:", self.reranker_model)
            self.reranker = CrossEncoder(self.reranker_model)
            self.modelo_reranker_usado = self.reranker_model
            self.motor = self.motor + "_reranker"
            print("Reranker carregado:", self.reranker_model)
        except Exception as e:
            print("Reranker indisponível. Seguindo sem reranker. Motivo:", repr(e))
            self.reranker = None

    def recuperar_um(self, query, top_k=TOP_K_FINAL):
        q = normalizar_texto(query)
        if not q:
            return []
        q_lex = self.vectorizer.transform([q])
        lex_scores = cosine_similarity(q_lex, self.X_lex).ravel()

        pool_n = min(max(top_k, self.candidate_pool), len(self.docs))

        # Correção pós-revisão: o pool de candidatos passa a ser a união dos
        # top-N lexicais com os top-N densos. Antes, o filtro inicial era
        # exclusivamente lexical, e fragmentos parafrásticos (semanticamente
        # próximos, lexicalmente distantes) nunca entravam no pool — relevante
        # agora que a base integrada tem ~18,5 mil fragmentos de treino.
        # Custo: o pool pode chegar a 2x candidate_pool no pior caso.
        dense_scores_all = None
        if self.embedder is not None and self.doc_embeddings is not None:
            try:
                q_emb = self.embedder.encode([q], normalize_embeddings=True, show_progress_bar=False)[0]
                dense_scores_all = np.dot(self.doc_embeddings, q_emb)
            except Exception as e:
                print("Falha em embedding da query; usando lexical. Motivo:", repr(e))
                dense_scores_all = None

        idx_lex = np.argsort(-lex_scores)[:pool_n]
        if dense_scores_all is not None:
            idx_den = np.argsort(-dense_scores_all)[:pool_n]
            idx_pool = np.array(sorted(set(idx_lex.tolist()) | set(idx_den.tolist())), dtype=int)
        else:
            idx_pool = idx_lex

        if dense_scores_all is not None:
            lex_norm = normalizar_score(lex_scores[idx_pool])
            dense_norm = normalizar_score(dense_scores_all[idx_pool])
            hybrid_scores = self.peso_lexical * lex_norm + self.peso_denso * dense_norm
        else:
            dense_scores_all = np.zeros_like(lex_scores)
            hybrid_scores = normalizar_score(lex_scores[idx_pool])

        # Reranking opcional dos candidatos.
        if self.reranker is not None:
            try:
                pares = [(q, self.docs[i]) for i in idx_pool]
                rerank_scores = np.asarray(self.reranker.predict(pares), dtype=float)
                final_scores = normalizar_score(rerank_scores)
            except Exception as e:
                print("Falha no reranker; usando score híbrido. Motivo:", repr(e))
                final_scores = hybrid_scores
        else:
            final_scores = hybrid_scores

        ordem_local = np.argsort(-final_scores)[:top_k]
        resultados = []
        for pos in ordem_local:
            i = int(idx_pool[pos])
            resultados.append({
                "score_final": float(final_scores[pos]),
                "score_lexical": float(lex_scores[i]),
                "score_denso": float(dense_scores_all[i]) if len(dense_scores_all) else np.nan,
                "id": self.ids[i],
                "tema": self.temas[i],
                "fragmento": self.fragmentos[i],
                "fonte_sugerida": self.fontes[i],
            })
        return resultados

    def aplicar(self, df, texto_col="texto_busca_v5", top_k=TOP_K_FINAL):
        linhas = []
        for _, row in df.iterrows():
            item = row.to_dict()
            res = self.recuperar_um(row.get(texto_col, ""), top_k=top_k)
            item["motor_rag_v5"] = self.motor
            item["modelo_embedding_v5"] = self.modelo_embedding_usado or "nenhum"
            item["modelo_reranker_v5"] = self.modelo_reranker_usado or "nenhum"
            for rank in range(1, top_k + 1):
                r = res[rank - 1] if rank - 1 < len(res) else {}
                item[f"score_final_top{rank}_v5"] = r.get("score_final", np.nan)
                item[f"score_lexical_top{rank}_v5"] = r.get("score_lexical", np.nan)
                item[f"score_denso_top{rank}_v5"] = r.get("score_denso", np.nan)
                item[f"id_top{rank}_v5"] = r.get("id", "")
                item[f"tema_top{rank}_v5"] = r.get("tema", "")
                item[f"fragmento_top{rank}_v5"] = r.get("fragmento", "")
                item[f"fonte_sugerida_top{rank}_v5"] = r.get("fonte_sugerida", "")
            linhas.append(item)
        return pd.DataFrame(linhas)

inicio = time.time()
motor_v5 = MotorRAGHibridoV5(base_rag)
print("Motor RAG v5:", motor_v5.motor)
print("Modelo embedding:", motor_v5.modelo_embedding_usado)
print("Modelo reranker:", motor_v5.modelo_reranker_usado)
print("Tempo de inicialização:", round(time.time() - inicio, 2), "s")


In [ ]:
# ============================================================
# 8. Executar RAG v5 nos três modos
# ============================================================

# Modo 1: teste inteiro.
teste_todo_v5 = motor_v5.aplicar(test, top_k=TOP_K_FINAL)

# Modo 2: irônicos reais ou preditos.
mask_ironicos = pd.Series(False, index=test.index)
if "label" in test.columns:
    mask_ironicos = mask_ironicos | (test["label"].astype(int) == 1)
if "bert_pred" in test.columns:
    mask_ironicos = mask_ironicos | (test["bert_pred"].fillna(0).astype(int) == 1)
ironicos_v5 = motor_v5.aplicar(test.loc[mask_ironicos].copy(), top_k=TOP_K_FINAL)

# Modo 3: baixa marcação superficial.
if "n_marc_sup" in test.columns:
    baixa_mask = test["n_marc_sup"].fillna(0).astype(float) <= 2
else:
    baixa_mask = pd.Series(True, index=test.index)
# Se houver BERT, prioriza casos preditos/rotulados como irônicos + baixa marcação.
baixa_mask = baixa_mask & mask_ironicos
baixa_v5 = motor_v5.aplicar(test.loc[baixa_mask].copy(), top_k=TOP_K_FINAL)

print("RAG v5 - teste todo:", teste_todo_v5.shape)
print("RAG v5 - irônicos reais/preditos:", ironicos_v5.shape)
print("RAG v5 - baixa marcação:", baixa_v5.shape)

teste_todo_v5.to_csv(OUT_DIR / "rag_news_teste_todo_v5_hibrido.csv", index=False)
ironicos_v5.to_csv(OUT_DIR / "rag_news_ironicos_v5_hibrido.csv", index=False)
baixa_v5.to_csv(OUT_DIR / "rag_news_baixa_marcacao_v5_hibrido.csv", index=False)

print("Arquivos RAG v5 salvos em:", OUT_DIR)


In [ ]:
# ============================================================
# 9. Saída integrada v5
# ============================================================

# Para a saída integrada, usamos o teste inteiro com os novos Top-5 v5.
triade_v5 = teste_todo_v5.copy()

# Explicação textual resumida da tríade.
def explicar_triade_v5(row):
    pred = row.get("bert_pred", pd.NA)
    zona = row.get("zona_adtc", "")
    n_marc = row.get("n_marc_sup", "")
    top1 = row.get("tema_top1_v5", "")
    motor = row.get("motor_rag_v5", "")
    if pd.isna(pred):
        pred_txt = "sem predição BERTimbau disponível"
    elif int(pred) == 1:
        pred_txt = "BERTimbau predisse ironia/sátira"
    else:
        pred_txt = "BERTimbau predisse não ironia"
    return (
        f"{pred_txt}; ADTC: {zona}; marcadores superficiais={n_marc}; "
        f"RAG v5 ({motor}) recuperou como contexto Top-1: {top1}."
    )

triade_v5["explicacao_triade_v5"] = triade_v5.apply(explicar_triade_v5, axis=1)
triade_v5.to_csv(OUT_DIR / "idpt_news_teste_triade_adtc_rag_v5_hibrido.csv", index=False)

print("Saída integrada v5:", triade_v5.shape)
display(triade_v5[["id_texto_original", "label", "bert_pred", "tipo_erro", "n_marc_sup", "zona_adtc", "motor_rag_v5", "tema_top1_v5", "score_final_top1_v5"]].head())


In [ ]:
# ============================================================
# 10. Amostra manual v5 em formato largo e legível
# ============================================================

# Mesma lógica da v4: foca em erros + baixa marcação + uma amostra complementar.
partes = []
if "tipo_erro" in triade_v5.columns:
    partes.append(triade_v5[triade_v5["tipo_erro"].isin(["FP", "FN"])])
if "n_marc_sup" in triade_v5.columns:
    partes.append(triade_v5[triade_v5["n_marc_sup"].fillna(0).astype(float) <= 2])
partes.append(triade_v5.sample(min(20, len(triade_v5)), random_state=42))

amostra_v5 = pd.concat(partes, ignore_index=True).drop_duplicates(subset=["id_texto_original"]).head(100)

cols_base = [c for c in ["id_texto_original", "label", "bert_pred", "tipo_erro", "n_marc_sup", "text", "texto_busca_v5", "marcadores", "zona_adtc"] if c in amostra_v5.columns]
cols_top = []
for k in range(1, TOP_K_FINAL + 1):
    cols_top.extend([
        f"score_final_top{k}_v5",
        f"id_top{k}_v5",
        f"tema_top{k}_v5",
        f"fragmento_top{k}_v5",
        f"fonte_sugerida_top{k}_v5",
        f"pertinencia_top{k}_manual_v5",
    ])
    amostra_v5[f"pertinencia_top{k}_manual_v5"] = ""

amostra_v5["observacao_manual_v5"] = ""
cols_saida = cols_base + [c for c in cols_top if c in amostra_v5.columns] + ["observacao_manual_v5"]
amostra_larga_v5 = amostra_v5[cols_saida].copy()
amostra_larga_v5.to_csv(OUT_DIR / "amostra_avaliacao_manual_rag_v5_hibrido.csv", index=False)

# Formato legível/longo: uma linha por contexto.
linhas = []
for _, row in amostra_v5.iterrows():
    for k in range(1, TOP_K_FINAL + 1):
        linhas.append({
            "id_texto_original": row.get("id_texto_original"),
            "label": row.get("label"),
            "bert_pred": row.get("bert_pred"),
            "tipo_erro": row.get("tipo_erro"),
            "n_marc_sup": row.get("n_marc_sup"),
            "texto": row.get("text", row.get("texto_busca_v5", "")),
            "rank_contexto": k,
            "score_final_v5": row.get(f"score_final_top{k}_v5"),
            "id_contexto_v5": row.get(f"id_top{k}_v5"),
            "tema_contexto_v5": row.get(f"tema_top{k}_v5"),
            "fragmento_contexto_v5": row.get(f"fragmento_top{k}_v5"),
            "fonte_sugerida_v5": row.get(f"fonte_sugerida_top{k}_v5"),
            "PREENCHER_pertinencia_0_1": "",
            "PREENCHER_tipo_pertinencia": "",
            "PREENCHER_observacao": "",
        })

amostra_legivel_v5 = pd.DataFrame(linhas)
amostra_legivel_v5.to_csv(OUT_DIR / "amostra_avaliacao_manual_rag_v5_hibrido_legivel.csv", index=False)

print("Amostra manual v5 larga:", amostra_larga_v5.shape)
print("Amostra manual v5 legível:", amostra_legivel_v5.shape)
display(amostra_legivel_v5.head(10))


In [ ]:
# ============================================================
# 11. Integrar avaliação manual v5/v6 preenchida — versão robusta
# ============================================================

TOP_K_FINAL = 5


def localizar_candidatos_manual_v5():
    bases = [
        "/kaggle/input",
        "/kaggle/working",
        "/kaggle/working/adtc_news",
        "/kaggle/working/adtc_news_v5",
        str(OUT_DIR),
        ".",
    ]
    padroes = [
        "*amostra_avaliacao_manual_rag_v5*preenchida*.csv",
        "*amostra_avaliacao_manual_rag_v5*.csv",
        "*avaliacao_manual_rag_v5*preenchida*.csv",
        "*avaliacao_manual_rag_v5*.csv",
    ]
    encontrados = []
    for base in bases:
        if not os.path.exists(base):
            continue
        for p in padroes:
            encontrados.extend(glob.glob(os.path.join(base, "**", p), recursive=True))
    return sorted(set([e for e in encontrados if os.path.isfile(e)]))


def contar_preenchimentos_manual(df):
    if "PREENCHER_pertinencia_0_1" in df.columns:
        s = pd.to_numeric(df["PREENCHER_pertinencia_0_1"], errors="coerce")
        return int(s.notna().sum())
    cols_v5 = [f"pertinencia_top{k}_manual_v5" for k in range(1, TOP_K_FINAL + 1) if f"pertinencia_top{k}_manual_v5" in df.columns]
    if cols_v5:
        tmp = df[cols_v5].apply(pd.to_numeric, errors="coerce")
        return int(tmp.notna().sum().sum())
    cols_old = [f"pertinencia_top{k}_manual" for k in range(1, TOP_K_FINAL + 1) if f"pertinencia_top{k}_manual" in df.columns]
    if cols_old:
        tmp = df[cols_old].apply(pd.to_numeric, errors="coerce")
        return int(tmp.notna().sum().sum())
    return 0

candidatos = localizar_candidatos_manual_v5()
metricas_rag_v5 = pd.DataFrame(columns=["metrica", "valor", "valor_percentual"])
triade_v5_com_manual = triade_v5.copy()
manual_wide = pd.DataFrame()

if candidatos:
    diagnostico = []
    for caminho in candidatos:
        try:
            df_tmp = ler_csv_robusto(caminho)
            diagnostico.append({"arquivo": caminho, "linhas": len(df_tmp), "preenchimentos": contar_preenchimentos_manual(df_tmp)})
        except Exception as e:
            diagnostico.append({"arquivo": caminho, "linhas": np.nan, "preenchimentos": -1, "erro": repr(e)})
    diagnostico_df = pd.DataFrame(diagnostico).sort_values("preenchimentos", ascending=False)
    print("Arquivos candidatos encontrados para avaliação manual:")
    display(diagnostico_df)

    arquivo_manual_v5 = diagnostico_df.iloc[0]["arquivo"]
    manual_v5 = ler_csv_robusto(arquivo_manual_v5)
    print("Arquivo escolhido:", arquivo_manual_v5)
    print("Preenchimentos detectados:", contar_preenchimentos_manual(manual_v5))

    if contar_preenchimentos_manual(manual_v5) == 0:
        print("Avaliação manual encontrada, mas ainda sem preenchimento. Métricas ficam vazias.")
    else:
        # Formato longo/legível.
        if "PREENCHER_pertinencia_0_1" in manual_v5.columns and "rank_contexto" in manual_v5.columns:
            manual_v5["PREENCHER_pertinencia_0_1"] = pd.to_numeric(manual_v5["PREENCHER_pertinencia_0_1"], errors="coerce")
            manual_wide = manual_v5.pivot_table(
                index="id_texto_original",
                columns="rank_contexto",
                values="PREENCHER_pertinencia_0_1",
                aggfunc="max",
            ).reset_index()
            manual_wide.columns = ["id_texto_original"] + [f"pertinencia_top{int(c)}_manual_v5" for c in manual_wide.columns[1:]]
        else:
            manual_wide = manual_v5.copy()
            for k in range(1, TOP_K_FINAL + 1):
                c_v5 = f"pertinencia_top{k}_manual_v5"
                c_old = f"pertinencia_top{k}_manual"
                if c_v5 not in manual_wide.columns and c_old in manual_wide.columns:
                    manual_wide[c_v5] = manual_wide[c_old]

        if "id_texto_original" not in manual_wide.columns:
            raise ValueError("A avaliação manual precisa ter id_texto_original.")

        cols_pert_v5 = [f"pertinencia_top{k}_manual_v5" for k in range(1, TOP_K_FINAL + 1) if f"pertinencia_top{k}_manual_v5" in manual_wide.columns]
        for c in cols_pert_v5:
            manual_wide[c] = pd.to_numeric(manual_wide[c], errors="coerce")

        avaliacao_real = manual_wide[manual_wide[cols_pert_v5].notna().any(axis=1)].copy()
        print("Linhas avaliadas manualmente:", len(avaliacao_real))

        def precision_at_k_manual(df, k):
            cols = [f"pertinencia_top{i}_manual_v5" for i in range(1, k + 1) if f"pertinencia_top{i}_manual_v5" in df.columns]
            return df[cols].mean(axis=1, skipna=True).mean()

        def hit_at_k_manual(df, k):
            cols = [f"pertinencia_top{i}_manual_v5" for i in range(1, k + 1) if f"pertinencia_top{i}_manual_v5" in df.columns]
            return df[cols].max(axis=1, skipna=True).mean()

        cobertura = len(avaliacao_real) / len(manual_wide)
        metricas_rag_v5 = pd.DataFrame({
            "metrica": ["Precision@1", "Precision@2", "Precision@5", "Hit@1", "Hit@2", "Hit@5", "Cobertura avaliada"],
            "valor": [
                precision_at_k_manual(avaliacao_real, 1),
                precision_at_k_manual(avaliacao_real, 2),
                precision_at_k_manual(avaliacao_real, 5),
                hit_at_k_manual(avaliacao_real, 1),
                hit_at_k_manual(avaliacao_real, 2),
                hit_at_k_manual(avaliacao_real, 5),
                cobertura,
            ],
        })
        metricas_rag_v5["valor_percentual"] = (metricas_rag_v5["valor"] * 100).round(2)

        # Integra ao arquivo triádico.
        triade_v5_com_manual = triade_v5.copy()
        triade_v5_com_manual["id_texto_original"] = pd.to_numeric(triade_v5_com_manual["id_texto_original"], errors="coerce").astype("Int64")
        manual_wide["id_texto_original"] = pd.to_numeric(manual_wide["id_texto_original"], errors="coerce").astype("Int64")
        triade_v5_com_manual = triade_v5_com_manual.merge(manual_wide[["id_texto_original"] + cols_pert_v5], on="id_texto_original", how="left")

        metricas_rag_v5.to_csv(OUT_DIR / "metricas_avaliacao_manual_rag_v5_hibrido.csv", index=False)
        manual_wide.to_csv(OUT_DIR / "amostra_avaliacao_manual_rag_v5_preenchida_conferida.csv", index=False)
        print("Métricas RAG v5/v6 calculadas:")
        display(metricas_rag_v5)
else:
    print("Avaliação manual v5 preenchida ainda não encontrada. O notebook gerou a amostra para preenchimento.")

triade_v5_com_manual.to_csv(OUT_DIR / "idpt_news_teste_triade_adtc_rag_v5_hibrido_com_avaliacao_manual.csv", index=False)
metricas_rag_v5.to_csv(OUT_DIR / "metricas_avaliacao_manual_rag_v5_hibrido.csv", index=False)

display(metricas_rag_v5)


In [ ]:
# ============================================================
# 12. Comparação v4 x v5 quando houver avaliação manual
#     Versão robusta: se o CSV de métricas v4 não for encontrado,
#     tenta reconstruir as métricas a partir da amostra manual v4.
# ============================================================

def calcular_metricas_rag_largo(df, prefixo_coluna="pertinencia_top", sufixo_coluna="_manual", top_k=5):
    """Calcula Precision@K e Hit@K para avaliação manual em formato largo."""
    df = df.copy()

    cols = []
    for k in range(1, top_k + 1):
        candidatos = [
            f"{prefixo_coluna}{k}{sufixo_coluna}",
            f"pertinencia_top{k}_manual",
            f"pertinencia_top{k}_manual_v4",
            f"pertinencia_top{k}_manual_v5",
        ]
        col = next((c for c in candidatos if c in df.columns), None)
        if col is not None:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            cols.append(col)

    if not cols:
        return pd.DataFrame(columns=["metrica", "valor", "valor_percentual"])

    def precision_at(k):
        use_cols = cols[:k]
        if not use_cols:
            return np.nan
        return df[use_cols].mean(axis=1, skipna=True).mean()

    def hit_at(k):
        use_cols = cols[:k]
        if not use_cols:
            return np.nan
        return df[use_cols].max(axis=1, skipna=True).mean()

    cobertura = df[cols].notna().any(axis=1).mean()

    out = pd.DataFrame({
        "metrica": [
            "Precision@1", "Precision@2", "Precision@5",
            "Hit@1", "Hit@2", "Hit@5", "Cobertura avaliada"
        ],
        "valor": [
            precision_at(1), precision_at(2), precision_at(min(5, len(cols))),
            hit_at(1), hit_at(2), hit_at(min(5, len(cols))),
            cobertura,
        ],
    })
    out["valor_percentual"] = (out["valor"] * 100).round(2)
    return out


arquivo_metricas_v4 = localizar_arquivo([
    "metricas_avaliacao_manual_rag_v4.csv",
    "metricas_avaliacao_manual_rag_v4*.csv",
], obrigatorio=False)

arquivo_manual_v4 = localizar_arquivo([
    "amostra_avaliacao_manual_rag_v4_conferida.csv",
    "amostra_avaliacao_manual_rag_v4_preenchida.csv",
    "amostra_avaliacao_manual_rag_v4*.csv",
    "amostra_avaliacao_manual_rag_v3_preenchida.csv",
    "amostra_avaliacao_manual_rag_v3*.csv",
], obrigatorio=False)

comparacao_rag = pd.DataFrame()
m4 = pd.DataFrame(columns=["metrica", "valor", "valor_percentual"])

if arquivo_metricas_v4 is not None:
    print("Métricas v4 encontradas:", arquivo_metricas_v4)
    m4 = ler_csv_robusto(arquivo_metricas_v4)
    if "valor" in m4.columns:
        m4["valor"] = pd.to_numeric(m4["valor"], errors="coerce")
    if "valor_percentual" not in m4.columns and "valor" in m4.columns:
        m4["valor_percentual"] = (m4["valor"] * 100).round(2)

elif arquivo_manual_v4 is not None:
    print("Métricas v4 não encontradas, mas amostra manual v4 foi encontrada:", arquivo_manual_v4)
    print("Reconstruindo métricas v4 a partir da amostra manual.")
    manual_v4 = ler_csv_robusto(arquivo_manual_v4)

    # Caso a amostra esteja no formato longo/legível.
    if "PREENCHER_pertinencia_0_1" in manual_v4.columns and "rank_contexto" in manual_v4.columns:
        manual_v4["PREENCHER_pertinencia_0_1"] = pd.to_numeric(
            manual_v4["PREENCHER_pertinencia_0_1"], errors="coerce"
        )
        manual_v4_largo = manual_v4.pivot_table(
            index="id_texto_original",
            columns="rank_contexto",
            values="PREENCHER_pertinencia_0_1",
            aggfunc="max",
        ).reset_index()
        manual_v4_largo.columns = ["id_texto_original"] + [
            f"pertinencia_top{int(c)}_manual" for c in manual_v4_largo.columns[1:]
        ]
        m4 = calcular_metricas_rag_largo(manual_v4_largo, top_k=TOP_K_FINAL)
    else:
        m4 = calcular_metricas_rag_largo(manual_v4, top_k=TOP_K_FINAL)

    if not m4.empty:
        m4.to_csv(OUT_DIR / "metricas_avaliacao_manual_rag_v4_reconstruidas.csv", index=False)
        print("Métricas v4 reconstruídas e salvas em:", OUT_DIR / "metricas_avaliacao_manual_rag_v4_reconstruidas.csv")

else:
    print("Métricas v4 e amostra manual v4 não encontradas.")
    print("A comparação v4 x v5 será omitida, mas o restante do notebook continua normalmente.")

if not m4.empty and not metricas_rag_v5.empty:
    m4 = m4.copy()
    m4["versao"] = "v4_tfidf"

    m5 = metricas_rag_v5.copy()
    if "valor" in m5.columns:
        m5["valor"] = pd.to_numeric(m5["valor"], errors="coerce")
    if "valor_percentual" not in m5.columns and "valor" in m5.columns:
        m5["valor_percentual"] = (m5["valor"] * 100).round(2)
    m5["versao"] = "v5_hibrido"

    cols_comp = ["metrica", "valor", "valor_percentual", "versao"]
    comparacao_rag = pd.concat([
        m4[[c for c in cols_comp if c in m4.columns]],
        m5[[c for c in cols_comp if c in m5.columns]],
    ], ignore_index=True)

    comparacao_rag.to_csv(OUT_DIR / "metricas_comparativas_rag_v4_v5.csv", index=False)
    print("Comparação v4 x v5 salva em:", OUT_DIR / "metricas_comparativas_rag_v4_v5.csv")
    display(comparacao_rag)
else:
    print("Comparação v4 x v5 não gerada nesta execução.")


In [ ]:
# ============================================================
# 13. Gráficos v5
# ============================================================

# 11.1 Matriz de confusão BERTimbau
if isinstance(cm, np.ndarray) and cm.shape == (2, 2) and np.isfinite(cm).all():
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred. não irônico", "Pred. irônico"], rotation=20, ha="right")
    ax.set_yticklabels(["Real não irônico", "Real irônico"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    ax.set_title("Matriz de confusão — BERTimbau")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "grafico_matriz_confusao_bertimbau_v5.png", dpi=200)
    plt.show()

# 11.2 Distribuição de zonas ADTC
if "zona_adtc" in triade_v5.columns:
    z = triade_v5["zona_adtc"].value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    z.plot(kind="bar", ax=ax)
    ax.set_title("Zonas ADTC — teste integrado v5")
    ax.set_xlabel("Zona")
    ax.set_ylabel("Quantidade")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "grafico_zonas_adtc_v5.png", dpi=200)
    plt.show()

# 11.3 Métricas RAG manual v5, se houver.
if not metricas_rag_v5.empty and metricas_rag_v5["valor_percentual"].notna().any():
    fig, ax = plt.subplots(figsize=(7, 4))
    metricas_rag_v5.set_index("metrica")["valor_percentual"].plot(kind="bar", ax=ax)
    ax.set_title("Métricas manuais do RAG v5")
    ax.set_xlabel("Métrica")
    ax.set_ylabel("%")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "grafico_metricas_rag_manual_v5.png", dpi=200)
    plt.show()

# 11.4 Comparação v4 x v5.
if not comparacao_rag.empty:
    pivot = comparacao_rag.pivot(index="metrica", columns="versao", values="valor_percentual")
    fig, ax = plt.subplots(figsize=(8, 4))
    pivot.plot(kind="bar", ax=ax)
    ax.set_title("Comparação RAG v4 x v5")
    ax.set_xlabel("Métrica")
    ax.set_ylabel("%")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "grafico_comparacao_rag_v4_v5.png", dpi=200)
    plt.show()


# Preparar BERTimbau simples para XAI

Esta seção configura o carregamento direto do BERTimbau simples pelo Hugging Face.

Modelo usado:

```text
neuralmind/bert-base-portuguese-cased
```

Esta versão **não procura checkpoint salvo**, **não exige modelo fine-tuned** e **não executa retreinamento**.


In [ ]:
# ============================================================
# Preparar BERTimbau simples para XAI
# ============================================================

CAMINHO_MODELO_BERT_XAI = "neuralmind/bert-base-portuguese-cased"
MODELO_BERT_XAI_ENCONTRADO = CAMINHO_MODELO_BERT_XAI

print("Modelo BERTimbau simples configurado para XAI:")
print(CAMINHO_MODELO_BERT_XAI)

# Esta versão não procura checkpoint e não retreina.
# O carregamento efetivo acontece na célula XAI via Hugging Face.


# 14. Módulo XAI neural para o BERTimbau

Esta seção adiciona SHAP, LIME, Integrated Gradients, attention visualization e análise de pesos internos.

In [ ]:

# ============================================================
# 14. Módulo XAI neural — SHAP, LIME, Integrated Gradients,
#     attention visualization e pesos internos
# ============================================================

import json
import math
from pathlib import Path

XAI_DIR = OUT_DIR / "xai_bertimbau"
XAI_DIR.mkdir(parents=True, exist_ok=True)

def encontrar_coluna_texto_xai(df):
    candidatos = [
        "text", "texto", "texto_modelo", "text_limpo", "texto_limpo",
        "noticia", "notícia", "content", "conteudo", "conteúdo"
    ]
    for c in candidatos:
        if c in df.columns:
            return c
    raise ValueError(f"Nenhuma coluna de texto encontrada para XAI. Colunas: {df.columns.tolist()}")

def encontrar_modelo_bert_finetuned():
    candidatos = []

    if CAMINHO_MODELO_BERT_XAI:
        candidatos.append(Path(CAMINHO_MODELO_BERT_XAI))

    for base in [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]:
        if base.exists():
            for pad in ["*modelo*bert*", "*bertimbau*", "*checkpoint*"]:
                candidatos.extend(base.glob(f"**/{pad}"))

    validos = []
    for c in candidatos:
        if not c.exists() or not c.is_dir():
            continue
        tem_config = (c / "config.json").exists()
        tem_pesos = (c / "pytorch_model.bin").exists() or (c / "model.safetensors").exists()
        if tem_config and tem_pesos:
            validos.append(c)

    validos = sorted(
        set(validos),
        key=lambda p: (0 if "checkpoint" in p.name.lower() else 1, len(str(p)))
    )
    return validos[0] if validos else None

# Base XAI: usa o teste integrado com ADTC, quando disponível.
if "triade_v5_com_manual" in globals():
    base_xai = triade_v5_com_manual.copy()
elif "triade_v5" in globals():
    base_xai = triade_v5.copy()
else:
    base_xai = test_pred_df.copy()

col_texto_xai = encontrar_coluna_texto_xai(base_xai)

for col in ["id_texto_original", "label", "bert_pred", "bert_conf", "bert_p_iro", "tipo_erro"]:
    if col not in base_xai.columns and col in test_pred_df.columns and len(test_pred_df) == len(base_xai):
        base_xai[col] = test_pred_df[col].values

amostras = []
if "tipo_erro" in base_xai.columns:
    for tipo in ["VP", "FP", "FN", "VN"]:
        sub = base_xai[base_xai["tipo_erro"].astype(str) == tipo].copy()
        if "bert_conf" in sub.columns:
            sub = sub.sort_values("bert_conf", ascending=False)
        amostras.append(sub.head(N_AMOSTRAS_XAI_POR_TIPO))
else:
    amostras.append(base_xai.head(N_AMOSTRAS_XAI_POR_TIPO * 4))

xai_amostra = pd.concat(amostras, ignore_index=True).drop_duplicates("id_texto_original", keep="first")
xai_amostra = xai_amostra.head(N_AMOSTRAS_XAI_POR_TIPO * 4).copy()
xai_amostra["texto_xai"] = xai_amostra[col_texto_xai].astype(str)
xai_amostra.to_csv(XAI_DIR / "xai_amostra_casos_bertimbau.csv", index=False)

modelo_xai_path = "neuralmind/bert-base-portuguese-cased"
print("Modelo XAI carregado:", modelo_xai_path)
display(xai_amostra[[c for c in ["id_texto_original", "label", "bert_pred", "bert_conf", "tipo_erro", "zona_adtc", "n_marc_sup"] if c in xai_amostra.columns]])

xai_status = pd.DataFrame([{
    "executar_xai_neural": EXECUTAR_XAI_NEURAL,
    "modelo_xai_path": str(modelo_xai_path) if modelo_xai_path is not None else None,
    "permitir_modelo_base_demo": PERMITIR_MODELO_BASE_APENAS_DEMO,
    "n_amostras_xai": len(xai_amostra),
}])
xai_status.to_csv(XAI_DIR / "xai_status_modelo.csv", index=False)

modelo_xai_disponivel = False
tokenizer_xai = None
model_xai = None
device_xai = None

if EXECUTAR_XAI_NEURAL:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
    except ImportError:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"])
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification

    if modelo_xai_path is not None:
        tokenizer_xai = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
        model_xai = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=2)
        modelo_xai_disponivel = True
        print("BERTimbau simples carregado para XAI:", BERT_MODEL_NAME)

    else:
        modelo_xai_disponivel = False
        print("XAI neural não executada: BERTimbau simples indisponível.")

if modelo_xai_disponivel:
    device_xai = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_xai.to(device_xai)
    model_xai.eval()

    def predict_proba_texts_xai(textos, batch_size=8):
        if isinstance(textos, np.ndarray):
            textos = textos.tolist()
        if isinstance(textos, str):
            textos = [textos]
        textos = [str(t) for t in textos]

        probs = []
        for i in range(0, len(textos), batch_size):
            batch = textos[i:i+batch_size]
            enc = tokenizer_xai(
                batch,
                truncation=True,
                padding=True,
                max_length=MAX_LEN_XAI,
                return_tensors="pt",
            )
            enc = {k: v.to(device_xai) for k, v in enc.items()}
            with torch.no_grad():
                out = model_xai(**enc)
                p = torch.softmax(out.logits, dim=1).detach().cpu().numpy()
            probs.append(p)
        return np.vstack(probs)

    probs_diag = predict_proba_texts_xai(xai_amostra["texto_xai"].tolist())
    xai_amostra["xai_model_pred"] = probs_diag.argmax(axis=1)
    xai_amostra["xai_model_p_iro"] = probs_diag[:, 1]
    xai_amostra["xai_model_conf"] = probs_diag.max(axis=1)
    if "bert_pred" in xai_amostra.columns:
        xai_amostra["xai_concorda_predicao_precomputada"] = (
            xai_amostra["xai_model_pred"].astype(int) == xai_amostra["bert_pred"].astype(int)
        )
    xai_amostra.to_csv(XAI_DIR / "xai_amostra_casos_bertimbau_com_pred_modelo.csv", index=False)
    display(xai_amostra[[c for c in [
        "id_texto_original", "label", "bert_pred", "bert_conf",
        "xai_model_pred", "xai_model_p_iro", "xai_model_conf",
        "xai_concorda_predicao_precomputada", "tipo_erro", "zona_adtc"
    ] if c in xai_amostra.columns]])

# ============================================================
# LIME
# ============================================================

lime_df = pd.DataFrame()

if modelo_xai_disponivel:
    try:
        try:
            from lime.lime_text import LimeTextExplainer
        except ImportError:
            import sys, subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lime"])
            from lime.lime_text import LimeTextExplainer

        explainer_lime = LimeTextExplainer(class_names=["nao_ironico", "ironico"])
        lime_rows = []

        for _, row in xai_amostra.iterrows():
            texto = row["texto_xai"]
            id_texto = row.get("id_texto_original", None)
            probs = predict_proba_texts_xai([texto])
            classe_alvo = int(probs.argmax(axis=1)[0])

            exp = explainer_lime.explain_instance(
                texto,
                predict_proba_texts_xai,
                labels=[classe_alvo],
                num_features=TOP_TOKENS_XAI,
                num_samples=LIME_NUM_SAMPLES,
            )

            for rank, (termo, peso) in enumerate(exp.as_list(label=classe_alvo), start=1):
                lime_rows.append({
                    "id_texto_original": id_texto,
                    "classe_explicada": classe_alvo,
                    "rank": rank,
                    "termo": termo,
                    "peso_lime": peso,
                    "label": row.get("label", np.nan),
                    "bert_pred_precomputado": row.get("bert_pred", np.nan),
                    "tipo_erro": row.get("tipo_erro", np.nan),
                    "zona_adtc": row.get("zona_adtc", np.nan),
                })

        lime_df = pd.DataFrame(lime_rows)
        lime_df.to_csv(XAI_DIR / "xai_lime_tokens.csv", index=False)
        print("LIME salvo:", XAI_DIR / "xai_lime_tokens.csv")
        display(lime_df.head(20))

    except Exception as e:
        warnings.warn(f"LIME não executado: {repr(e)}")

# ============================================================
# Integrated Gradients
# ============================================================

ig_df = pd.DataFrame()

if modelo_xai_disponivel:
    try:
        try:
            from captum.attr import LayerIntegratedGradients
        except ImportError:
            import sys, subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "captum"])
            from captum.attr import LayerIntegratedGradients

        def forward_ig(input_ids, attention_mask, token_type_ids, target_class):
            kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
            if token_type_ids is not None:
                kwargs["token_type_ids"] = token_type_ids
            out = model_xai(**kwargs)
            return out.logits[:, target_class]

        lig = LayerIntegratedGradients(forward_ig, model_xai.bert.embeddings)
        ig_rows = []

        for _, row in xai_amostra.iterrows():
            texto = row["texto_xai"]
            id_texto = row.get("id_texto_original", None)

            enc = tokenizer_xai(
                texto,
                truncation=True,
                padding="max_length",
                max_length=MAX_LEN_XAI,
                return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device_xai)
            attention_mask = enc["attention_mask"].to(device_xai)
            token_type_ids = enc.get("token_type_ids", None)
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device_xai)

            probs = predict_proba_texts_xai([texto])
            classe_alvo = int(probs.argmax(axis=1)[0])

            baseline_ids = torch.full_like(input_ids, tokenizer_xai.pad_token_id)
            baseline_ids[:, 0] = input_ids[:, 0]
            sep_id = tokenizer_xai.sep_token_id
            if sep_id is not None:
                sep_positions = (input_ids == sep_id).nonzero(as_tuple=False)
                for pos in sep_positions:
                    baseline_ids[pos[0], pos[1]] = sep_id

            attributions, delta = lig.attribute(
                inputs=input_ids,
                baselines=baseline_ids,
                additional_forward_args=(attention_mask, token_type_ids, classe_alvo),
                n_steps=IG_N_STEPS,
                return_convergence_delta=True,
            )

            scores = attributions.sum(dim=-1).squeeze(0).detach().cpu().numpy()
            ids = input_ids.squeeze(0).detach().cpu().numpy()
            mask = attention_mask.squeeze(0).detach().cpu().numpy()
            tokens = tokenizer_xai.convert_ids_to_tokens(ids)

            for idx_tok, (tok, score, m) in enumerate(zip(tokens, scores, mask)):
                if int(m) == 0:
                    continue
                if tok in [tokenizer_xai.cls_token, tokenizer_xai.sep_token, tokenizer_xai.pad_token]:
                    continue
                ig_rows.append({
                    "id_texto_original": id_texto,
                    "classe_explicada": classe_alvo,
                    "posicao_token": idx_tok,
                    "token": tok,
                    "score_ig": float(score),
                    "abs_score_ig": float(abs(score)),
                    "label": row.get("label", np.nan),
                    "bert_pred_precomputado": row.get("bert_pred", np.nan),
                    "tipo_erro": row.get("tipo_erro", np.nan),
                    "zona_adtc": row.get("zona_adtc", np.nan),
                })

        ig_df = pd.DataFrame(ig_rows)
        if not ig_df.empty:
            ig_df = ig_df.sort_values(["id_texto_original", "abs_score_ig"], ascending=[True, False])
        ig_df.to_csv(XAI_DIR / "xai_integrated_gradients_tokens.csv", index=False)
        print("Integrated Gradients salvo:", XAI_DIR / "xai_integrated_gradients_tokens.csv")
        display(ig_df.head(20))

    except Exception as e:
        warnings.warn(f"Integrated Gradients não executado: {repr(e)}")

# ============================================================
# Attention visualization
# ============================================================

att_df = pd.DataFrame()

if modelo_xai_disponivel:
    try:
        att_rows = []
        for _, row in xai_amostra.iterrows():
            texto = row["texto_xai"]
            id_texto = row.get("id_texto_original", None)

            enc = tokenizer_xai(
                texto,
                truncation=True,
                padding=False,
                max_length=MAX_LEN_XAI,
                return_tensors="pt",
            )
            enc = {k: v.to(device_xai) for k, v in enc.items()}

            with torch.no_grad():
                out = model_xai(**enc, output_attentions=True)

            last_att = out.attentions[-1][0].detach().cpu().numpy()
            cls_media = last_att[:, 0, :].mean(axis=0)

            ids = enc["input_ids"].squeeze(0).detach().cpu().numpy()
            tokens = tokenizer_xai.convert_ids_to_tokens(ids)
            probs = torch.softmax(out.logits, dim=1).detach().cpu().numpy()
            classe_alvo = int(probs.argmax(axis=1)[0])

            for idx_tok, (tok, att) in enumerate(zip(tokens, cls_media)):
                if tok in [tokenizer_xai.cls_token, tokenizer_xai.sep_token, tokenizer_xai.pad_token]:
                    continue
                att_rows.append({
                    "id_texto_original": id_texto,
                    "classe_predita_modelo_xai": classe_alvo,
                    "posicao_token": idx_tok,
                    "token": tok,
                    "attention_cls_media_ultima_camada": float(att),
                    "label": row.get("label", np.nan),
                    "bert_pred_precomputado": row.get("bert_pred", np.nan),
                    "tipo_erro": row.get("tipo_erro", np.nan),
                    "zona_adtc": row.get("zona_adtc", np.nan),
                })

        att_df = pd.DataFrame(att_rows)
        if not att_df.empty:
            att_df = att_df.sort_values(["id_texto_original", "attention_cls_media_ultima_camada"], ascending=[True, False])
        att_df.to_csv(XAI_DIR / "xai_attention_cls_tokens.csv", index=False)
        print("Attention salvo:", XAI_DIR / "xai_attention_cls_tokens.csv")
        display(att_df.head(20))

    except Exception as e:
        warnings.warn(f"Attention visualization não executada: {repr(e)}")

# ============================================================
# SHAP
# ============================================================

shap_df = pd.DataFrame()

if modelo_xai_disponivel:
    try:
        try:
            import shap
        except ImportError:
            import sys, subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
            import shap

        textos_shap = xai_amostra["texto_xai"].head(SHAP_MAX_AMOSTRAS).tolist()
        ids_shap = xai_amostra["id_texto_original"].head(SHAP_MAX_AMOSTRAS).tolist()

        masker = shap.maskers.Text(tokenizer_xai)
        explainer_shap = shap.Explainer(
            predict_proba_texts_xai,
            masker,
            output_names=["nao_ironico", "ironico"]
        )

        shap_values = explainer_shap(textos_shap)
        shap_rows = []

        for i, id_texto in enumerate(ids_shap):
            probs = predict_proba_texts_xai([textos_shap[i]])
            classe_alvo = int(probs.argmax(axis=1)[0])

            tokens = list(shap_values.data[i])
            values = shap_values.values[i]

            if values.ndim == 2:
                scores = values[:, classe_alvo]
            else:
                scores = values

            ordenados = sorted(
                zip(tokens, scores),
                key=lambda x: abs(float(x[1])),
                reverse=True
            )[:TOP_TOKENS_XAI]

            for rank, (tok, score) in enumerate(ordenados, start=1):
                shap_rows.append({
                    "id_texto_original": id_texto,
                    "classe_explicada": classe_alvo,
                    "rank": rank,
                    "token": str(tok),
                    "score_shap": float(score),
                    "abs_score_shap": float(abs(score)),
                })

        shap_df = pd.DataFrame(shap_rows)
        if not shap_df.empty:
            shap_df = shap_df.merge(
                xai_amostra[[c for c in ["id_texto_original", "label", "bert_pred", "tipo_erro", "zona_adtc"] if c in xai_amostra.columns]],
                on="id_texto_original",
                how="left"
            )

        shap_df.to_csv(XAI_DIR / "xai_shap_tokens.csv", index=False)
        print("SHAP salvo:", XAI_DIR / "xai_shap_tokens.csv")
        display(shap_df.head(20))

    except Exception as e:
        warnings.warn(f"SHAP não executado: {repr(e)}")

# ============================================================
# Análise de pesos internos
# ============================================================

pesos_df = pd.DataFrame()

if modelo_xai_disponivel:
    try:
        pesos_rows = []

        if hasattr(model_xai, "classifier") and hasattr(model_xai.classifier, "weight"):
            w = model_xai.classifier.weight.detach().cpu().numpy()
            for classe in range(w.shape[0]):
                pesos_rows.append({
                    "componente": "classifier_head",
                    "classe": classe,
                    "parametro": "weight_norm_l2",
                    "valor": float(np.linalg.norm(w[classe])),
                })
            if w.shape[0] == 2:
                pesos_rows.append({
                    "componente": "classifier_head",
                    "classe": "1_minus_0",
                    "parametro": "diff_weight_norm_l2",
                    "valor": float(np.linalg.norm(w[1] - w[0])),
                })

        if hasattr(model_xai, "bert"):
            for i, layer in enumerate(model_xai.bert.encoder.layer):
                att = layer.attention.self
                for nome, modulo in [("query", att.query), ("key", att.key), ("value", att.value)]:
                    pesos_rows.append({
                        "componente": f"bert_layer_{i}_attention_{nome}",
                        "classe": "NA",
                        "parametro": "weight_norm_l2",
                        "valor": float(torch.norm(modulo.weight.detach()).cpu().item()),
                    })

        pesos_df = pd.DataFrame(pesos_rows)
        pesos_df.to_csv(XAI_DIR / "xai_pesos_internos_transformer.csv", index=False)
        print("Pesos internos salvos:", XAI_DIR / "xai_pesos_internos_transformer.csv")
        display(pesos_df.head(20))

    except Exception as e:
        warnings.warn(f"Análise de pesos internos não executada: {repr(e)}")

# ============================================================
# Síntese XAI + ADTC
# ============================================================

def top_tokens_por_id(df, id_col, token_col, score_col, prefixo, topn=10):
    if df is None or df.empty or id_col not in df.columns:
        return pd.DataFrame(columns=[id_col, f"{prefixo}_top_tokens"])
    tmp = df.copy()
    tmp["abs_score"] = tmp[score_col].abs() if score_col in tmp.columns else 0
    tmp = tmp.sort_values([id_col, "abs_score"], ascending=[True, False])
    linhas = []
    for id_texto, g in tmp.groupby(id_col):
        toks = []
        for _, r in g.head(topn).iterrows():
            toks.append(f"{r[token_col]} ({r[score_col]:.4f})")
        linhas.append({id_col: id_texto, f"{prefixo}_top_tokens": "; ".join(toks)})
    return pd.DataFrame(linhas)

sintese_xai = xai_amostra.copy()

if not lime_df.empty:
    sintese_xai = sintese_xai.merge(top_tokens_por_id(lime_df, "id_texto_original", "termo", "peso_lime", "lime"), on="id_texto_original", how="left")
if not ig_df.empty:
    sintese_xai = sintese_xai.merge(top_tokens_por_id(ig_df, "id_texto_original", "token", "score_ig", "ig"), on="id_texto_original", how="left")
if not att_df.empty:
    sintese_xai = sintese_xai.merge(top_tokens_por_id(att_df, "id_texto_original", "token", "attention_cls_media_ultima_camada", "attention"), on="id_texto_original", how="left")
if not shap_df.empty:
    sintese_xai = sintese_xai.merge(top_tokens_por_id(shap_df, "id_texto_original", "token", "score_shap", "shap"), on="id_texto_original", how="left")

sintese_cols = [c for c in [
    "id_texto_original", "label", "bert_pred", "bert_conf", "bert_p_iro",
    "tipo_erro", "zona_adtc", "n_marc_sup", "marcadores",
    "lime_top_tokens", "ig_top_tokens", "attention_top_tokens", "shap_top_tokens"
] if c in sintese_xai.columns]

sintese_xai[sintese_cols].to_csv(XAI_DIR / "xai_sintese_adtc_bertimbau.csv", index=False)
display(sintese_xai[sintese_cols].head(20))

xai_relatorio = f"""
# Relatório XAI — BERTimbau + ADTC + RAG

## Escopo

Este relatório documenta a camada XAI adicionada ao pipeline v6 sem retreinamento.

## Modelo usado para XAI

- BERTimbau simples carregado: **{modelo_xai_path if modelo_xai_path is not None else 'não encontrado'}**
- XAI neural executada: **{modelo_xai_disponivel}**
- Modo BERTimbau simples: **{PERMITIR_MODELO_BASE_APENAS_DEMO}**

## Métodos previstos

- LIME: {'executado' if not lime_df.empty else 'não executado'}
- SHAP: {'executado' if not shap_df.empty else 'não executado'}
- Integrated Gradients: {'executado' if not ig_df.empty else 'não executado'}
- Attention visualization: {'executado' if not att_df.empty else 'não executado'}
- Análise de pesos internos: {'executado' if not pesos_df.empty else 'não executado'}

## Interpretação metodológica

A ADTC já funciona como camada de explicabilidade discursiva. O módulo XAI neural acrescenta explicações em nível de modelo.

A leitura recomendada é:

> O pipeline combina explicabilidade discursiva, por meio da ADTC, com explicabilidade neural local, por meio de LIME, SHAP, Integrated Gradients, attention e análise de pesos internos.

## Cuidado metodológico

SHAP, LIME, Integrated Gradients e attention só devem ser usados como evidência empírica quando o modelo carregado for o BERTimbau simples carregado nesta execução analisadas. Caso contrário, os resultados são apenas demonstração técnica.
"""

(XAI_DIR / "RELATORIO_XAI_BERTIMBAU_ADTC_RAG.md").write_text(xai_relatorio, encoding="utf-8")
print("Relatório XAI salvo:", XAI_DIR / "RELATORIO_XAI_BERTIMBAU_ADTC_RAG.md")


In [ ]:
# ============================================================
# 15. Planilha final, relatório e ZIP — v6-XAI sem retreinamento
# ============================================================

try:
    diagnostico_antivazamento
except NameError:
    diagnostico_antivazamento = pd.DataFrame({"item": [], "valor": []})

try:
    diagnostico_split
except NameError:
    diagnostico_split = pd.DataFrame({"item": [], "valor": []})

try:
    diagnostico_predicoes
except NameError:
    diagnostico_predicoes = pd.DataFrame({"item": [], "valor": []})

try:
    quadro_taxonomia
except NameError:
    quadro_taxonomia = pd.DataFrame()

try:
    mapa_marcador_taxonomia_df
except NameError:
    mapa_marcador_taxonomia_df = pd.DataFrame()

try:
    evidencias_taxonomia_textos
except NameError:
    evidencias_taxonomia_textos = pd.DataFrame()

try:
    taxonomia_resumo_categorias
except NameError:
    taxonomia_resumo_categorias = pd.DataFrame()


resumo_geral = pd.DataFrame({
    "item": [
        "versao_pipeline",
        "dataset",
        "total_teste",
        "total_base_rag",
        "usa_predicoes_precomputadas",
        "bertimbau_retreinado_no_notebook",
        "bert_epochs_original",
        "max_len_bert_original",
        "treino_bruto_original",
        "treino_pos_trava_antivazamento",
        "vazamentos_exatos_treino_teste_removidos",
        "overlap_hash_pos_trava",
        "hashes_comuns_treino_validacao",
        "taxonomia_adtc_integrada",
        "taxonomia_total_recursos",
        "taxonomia_total_categorias",
        "taxonomia_doc_fonte",
        "motor_rag_hibrido",
        "modelo_embedding",
        "modelo_reranker",
        "rag_teste_todo",
        "rag_ironicos",
        "rag_baixa_marcacao",
    ],
    "valor": [
        "v7_xai_sem_retreino_bertimbau_simples_predicoes_bertimbau",
        "IDPT News integral",
        len(test),
        len(base_rag),
        "True",
        str(RETREINAR_BERTIMBAU),
        BERT_EPOCHS,
        MAX_LEN_BERT,
        globals().get("n_treino_antes_trava", "NA"),
        globals().get("n_treino_depois_trava", "NA"),
        globals().get("n_vazamento_exato", "NA"),
        globals().get("n_overlap_pos", "NA"),
        globals().get("n_overlap_train_val", "NA"),
        "True" if not quadro_taxonomia.empty else "False",
        len(quadro_taxonomia) if not quadro_taxonomia.empty else 0,
        quadro_taxonomia["categoria_num"].nunique() if not quadro_taxonomia.empty and "categoria_num" in quadro_taxonomia.columns else 0,
        DOCX_TAXONOMIA_ADTC,
        motor_v5.motor,
        motor_v5.modelo_embedding_usado or "nenhum",
        motor_v5.modelo_reranker_usado or "nenhum",
        len(teste_todo_v5),
        len(ironicos_v5),
        len(baixa_v5),
    ]
})

zonas = triade_v5["zona_adtc"].value_counts(dropna=False).reset_index()
zonas.columns = ["zona_adtc", "n"]

tipos_erro = triade_v5["tipo_erro"].value_counts(dropna=False).reset_index() if "tipo_erro" in triade_v5.columns else pd.DataFrame()
if not tipos_erro.empty:
    tipos_erro.columns = ["tipo_erro", "n"]

# Salva aliases com nome v7_xai_sem_retreino_bertimbau_simples.
try:
    triade_v5.to_csv(OUT_DIR / "idpt_news_teste_triade_adtc_rag_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv", index=False)
    teste_todo_v5.to_csv(OUT_DIR / "rag_news_teste_todo_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv", index=False)
    ironicos_v5.to_csv(OUT_DIR / "rag_news_ironicos_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv", index=False)
    baixa_v5.to_csv(OUT_DIR / "rag_news_baixa_marcacao_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv", index=False)
    metricas_rag_v5.to_csv(OUT_DIR / "metricas_avaliacao_manual_rag_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv", index=False)
except Exception as e:
    warnings.warn(f"Não foi possível salvar aliases v7_xai_sem_retreino_bertimbau_simples: {repr(e)}")

xlsx_path = OUT_DIR / "planilha_final_adtc_rag_news_v7_xai_sem_retreino_bertimbau_simples_hibrido.xlsx"
try:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        resumo_geral.to_excel(writer, sheet_name="Resumo", index=False)
        diagnostico_antivazamento.to_excel(writer, sheet_name="AntiVazamento", index=False)
        diagnostico_split.to_excel(writer, sheet_name="Split_train_val", index=False)
        diagnostico_predicoes.to_excel(writer, sheet_name="Predicoes_BERT", index=False)
        if not quadro_taxonomia.empty:
            quadro_taxonomia.to_excel(writer, sheet_name="Taxonomia_ADTC", index=False)
        if not taxonomia_resumo_categorias.empty:
            taxonomia_resumo_categorias.to_excel(writer, sheet_name="Taxonomia_resumo", index=False)
        if not mapa_marcador_taxonomia_df.empty:
            mapa_marcador_taxonomia_df.to_excel(writer, sheet_name="Mapa_marcador_taxonomia", index=False)
        if not evidencias_taxonomia_textos.empty:
            evidencias_taxonomia_textos.head(1000).to_excel(writer, sheet_name="Evid_taxonomia_textos", index=False)
        metricas_bert.to_excel(writer, sheet_name="Metricas_BERT", index=False)
        try:
            metricas_bert_precomputado.to_excel(writer, sheet_name="Metricas_BERT_original", index=False)
        except Exception:
            pass
        zonas.to_excel(writer, sheet_name="Zonas_ADTC", index=False)
        tipos_erro.to_excel(writer, sheet_name="Tipos_erro", index=False)
        metricas_rag_v5.to_excel(writer, sheet_name="Metricas_RAG", index=False)
        comparacao_rag.to_excel(writer, sheet_name="Comparacao_v4_v5", index=False)
        triade_v5.head(300).to_excel(writer, sheet_name="Teste_integrado", index=False)
        amostra_larga_v5.to_excel(writer, sheet_name="Amostra_manual", index=False)
        teste_todo_v5.head(300).to_excel(writer, sheet_name="RAG_teste_todo", index=False)
        ironicos_v5.head(300).to_excel(writer, sheet_name="RAG_ironicos", index=False)
        baixa_v5.head(300).to_excel(writer, sheet_name="RAG_baixa", index=False)
    print("Planilha salva:", xlsx_path)
except Exception as e:
    warnings.warn(f"Não foi possível salvar XLSX ({repr(e)}). Salvando CSVs separados.")
    xlsx_path = None
    resumo_geral.to_csv(OUT_DIR / "resumo_geral_v7_xai_sem_retreino_bertimbau_simples.csv", index=False)
    metricas_bert.to_csv(OUT_DIR / "metricas_bert_v7_xai_sem_retreino_bertimbau_simples.csv", index=False)
    zonas.to_csv(OUT_DIR / "zonas_adtc_v7_xai_sem_retreino_bertimbau_simples.csv", index=False)
    metricas_rag_v5.to_csv(OUT_DIR / "metricas_rag_v7_xai_sem_retreino_bertimbau_simples.csv", index=False)

relatorio = f"""
# Relatório final — ADTC + BERTimbau simples + predições pré-computadas + RAG Híbrido + XAI v7 sem retreinamento

Data de geração: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Escopo

Este notebook executa o pipeline integrado para o **IDPT News integral**, com trava anti-vazamento entre treino e teste, integração com a camada ADTC e aplicação do RAG híbrido preservado da v5.

A diferença central em relação à v6 integral é que esta versão **não retreina o BERTimbau**. Ela utiliza as predições já geradas pelo BERTimbau clássico.

A versão deve ser descrita como:

> **BERTimbau pré-computado no IDPT integral + trava anti-vazamento diagnóstica + ADTC + RAG híbrido BGE-M3/reranker.**

## Trava anti-vazamento

A trava foi aplicada como diagnóstico de segurança experimental sobre os arquivos `training_news.csv` e `test_news_com_rotulo.csv`. O notebook calcula um hash normalizado dos textos, verifica se algum texto do teste aparece no treino e interrompe a execução se ainda houver sobreposição após a remoção.

- Treino bruto original: **{globals().get('n_treino_antes_trava', 'NA')}**
- Teste: **{len(test_raw) if 'test_raw' in globals() else len(test)}**
- Linhas removidas do treino por vazamento exato com o teste: **{globals().get('n_vazamento_exato', 'NA')}**
- Treino após trava: **{globals().get('n_treino_depois_trava', 'NA')}**
- Overlap de hash treino/teste após trava: **{globals().get('n_overlap_pos', 'NA')}**
- Overlap de hash treino/validação no diagnóstico: **{globals().get('n_overlap_train_val', 'NA')}**

## Taxonomia ADTC incorporada

O notebook incorpora o **Quadro-Síntese da Taxonomia dos Recursos Irônicos no Português Brasileiro**, com **{len(quadro_taxonomia) if not quadro_taxonomia.empty else 'NA'} recursos**, distribuídos em **{quadro_taxonomia["categoria_num"].nunique() if not quadro_taxonomia.empty and "categoria_num" in quadro_taxonomia.columns else 'NA'} categorias**.

A taxonomia é usada para explicar a classificação discursiva da ironia. Cada marcador ADTC encontrado no texto é mapeado, quando possível, para um ou mais recursos do quadro. Assim, a saída final não apresenta apenas `irônico` ou `não irônico`; ela registra também os recursos taxonômicos mobilizados, as categorias discursivas correspondentes e a explicação classificatória.

As categorias incorporadas são:

{taxonomia_resumo_categorias.to_markdown(index=False) if not taxonomia_resumo_categorias.empty else 'Taxonomia não carregada.'}

Arquivos taxonômicos gerados:

- `quadro_taxonomia_ironia_pb_integrado.csv`
- `mapa_marcadores_adtc_taxonomia.csv`
- `evidencias_taxonomia_por_texto_v6_xai.csv`
- `idpt_news_teste_com_taxonomia_adtc_v6_xai.csv`

## Dados e predições BERTimbau

- BERTimbau simples carregado para XAI: **{CAMINHO_MODELO_BERT_XAI if 'CAMINHO_MODELO_BERT_XAI' in globals() else 'NA'}**
- XAI neural executada: **{globals().get('modelo_xai_disponivel', False)}**


- BERTimbau retreinado no notebook: **{RETREINAR_BERTIMBAU}**
- Predições pré-computadas usadas: **True**
- Total de textos no teste: **{len(test)}**
- Fragmentos na base RAG: **{len(base_rag)}**
- RAG no teste inteiro: **{len(teste_todo_v5)}**
- RAG em irônicos reais ou preditos: **{len(ironicos_v5)}**
- RAG em baixa marcação superficial: **{len(baixa_v5)}**

## Motor RAG

- Motor usado: **{motor_v5.motor}**
- Modelo de embedding: **{motor_v5.modelo_embedding_usado or 'nenhum'}**
- Modelo de reranker: **{motor_v5.modelo_reranker_usado or 'nenhum'}**

Se o motor aparece como `tfidf_fallback`, o ambiente não conseguiu carregar os modelos neurais. Nesse caso, o resultado continua válido como baseline lexical, mas não como RAG semântico robusto.

## Interpretação metodológica

Esta versão preserva as melhorias da v6 relacionadas à organização do pipeline, à verificação anti-vazamento, à integração ADTC e ao RAG híbrido. No entanto, ela não executa novo treinamento supervisionado. As zonas ADTC e os subconjuntos enviados ao RAG são produzidos a partir das predições já existentes do BERTimbau clássico.

Essa configuração é útil quando se deseja separar a avaliação do classificador neural da avaliação do pipeline discursivo-contextual. O BERTimbau fornece uma saída neural estável; a ADTC explica as marcas e zonas interpretativas; e o RAG recupera contextos para casos que exigem memória discursiva.

## Saídas principais

- `idpt_news_predicoes_teste_precomputado_v7_xai_sem_retreino_bertimbau_simples.csv`
- `idpt_news_predicoes_validacao_precomputado_v7_xai_sem_retreino_bertimbau_simples.csv`
- `diagnostico_antivazamento_treino_teste_v7_xai_sem_retreino_bertimbau_simples.csv`
- `diagnostico_split_train_val_v7_xai_sem_retreino_bertimbau_simples.csv`
- `diagnostico_predicoes_bertimbau_precomputadas_v7_xai_sem_retreino_bertimbau_simples.csv`
- `idpt_news_teste_triade_adtc_rag_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv`
- `rag_news_teste_todo_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv`
- `rag_news_ironicos_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv`
- `rag_news_baixa_marcacao_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv`
- `metricas_avaliacao_manual_rag_v7_xai_sem_retreino_bertimbau_simples_hibrido.csv`
- `planilha_final_adtc_rag_news_v7_xai_sem_retreino_bertimbau_simples_hibrido.xlsx`
- `quadro_taxonomia_ironia_pb_integrado.csv`
- `mapa_marcadores_adtc_taxonomia.csv`
- `evidencias_taxonomia_por_texto_v6_xai.csv`
- `idpt_news_teste_com_taxonomia_adtc_v6_xai.csv`
"""

rel_path = OUT_DIR / "RELATORIO_FINAL_ADTC_RAG_NEWS_V7_XAI_SEM_RETREINO_BERTIMBAU_SIMPLES.md"
rel_path.write_text(relatorio, encoding="utf-8")
print("Relatório salvo:", rel_path)

zip_saida = Path("/kaggle/working/saidas_finais_adtc_rag_news_v7_xai_sem_retreino_bertimbau_simples.zip") if os.path.exists("/kaggle/working") else Path("saidas_finais_adtc_rag_news_v7_xai_sem_retreino_bertimbau_simples.zip")
with zipfile.ZipFile(zip_saida, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.glob("**/*"):
        if p.is_file():
            z.write(p, p.relative_to(OUT_DIR))
print("ZIP final salvo:", zip_saida)

try:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_saida)))
except Exception:
    pass


## Como interpretar esta versão

Esta é a versão **v7-XAI sem retreinamento com BERTimbau simples corrigida**.

Ela não treina nem retreina o BERTimbau.

Ela usa:

- predições BERTimbau clássico já geradas;
- BERTimbau simples carregado via `neuralmind/bert-base-portuguese-cased`;
- Taxonomia ADTC;
- RAG híbrido;
- XAI neural local.

Formulação recomendada:

> **v7-XAI sem retreinamento = predições BERTimbau clássico + BERTimbau simples carregado + Taxonomia ADTC + RAG híbrido + XAI neural local.**


# 16. RAG contextual aprimorado

A avaliação manual mostrou que o RAG anterior retornava candidatos para todos os itens, mas com baixa precisão contextual. Nesta etapa, o objetivo não é alterar a predição do BERTimbau, mas melhorar a busca de contexto para auditoria.

A melhoria implementada aqui amplia a base de busca:

1. mantém a base curada de fragmentos contextuais;
2. acrescenta textos do conjunto de treino do IDPT como memória contextual sem usar seus rótulos;
3. constrói uma consulta enriquecida por texto, combinando início da notícia, palavras-chave, marcadores ADTC, zona interpretativa e categorias taxonômicas;
4. gera campos de consulta externa sugerida para pesquisa manual posterior;
5. preserva a saída do classificador neural e acrescenta evidências contextuais.

Cuidado metodológico:

> A base de treino é usada apenas como memória textual de comparação temática. Os rótulos do treino não são usados na recuperação contextual.


In [ ]:

# ============================================================
# 16. RAG contextual aprimorado — base expandida + consulta enriquecida
# ============================================================

STOP_PT = {
    "a","o","as","os","um","uma","uns","umas","de","do","da","dos","das","em","no","na","nos","nas",
    "por","para","com","sem","sob","sobre","entre","até","após","antes","e","ou","mas","que","se",
    "ao","aos","à","às","como","mais","menos","muito","muita","muitos","muitas","também","já","foi",
    "ser","são","era","eram","ter","tem","têm","há","não","sim","sua","seu","suas","seus","ele","ela",
    "eles","elas","isso","isto","aquele","aquela","aqueles","aquelas","este","esta","estes","estas",
    "diz","disse","afirmou","segundo","contra","durante","onde","quando","porque","ainda","apenas",
    "anos","ano","dia","dias","meses","mês"
}

def texto_curto(s, n=1400):
    s = str(s) if not pd.isna(s) else ""
    s = re.sub(r"\s+", " ", s).strip()
    return s[:n]

def extrair_palavras_chave_simples(texto, max_terms=18):
    texto = str(texto)
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9][A-Za-zÀ-ÖØ-öø-ÿ0-9_-]{3,}", texto)
    freq = {}
    ordem = []
    for t in tokens:
        tl = t.lower()
        if tl in STOP_PT:
            continue
        if tl not in freq:
            ordem.append(tl)
            freq[tl] = 0
        freq[tl] += 1
    termos = sorted(freq.keys(), key=lambda x: (-freq[x], ordem.index(x)))[:max_terms]
    return termos

def extrair_entidades_simples(texto, max_ent=10):
    texto = str(texto)
    candidatos = re.findall(r"\b[A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-Za-zÁÉÍÓÚÂÊÔÃÕÇáéíóúâêôãõç]+(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-Za-zÁÉÍÓÚÂÊÔÃÕÇáéíóúâêôãõç]+){0,3}", texto)
    vistos, ents = set(), []
    for c in candidatos:
        c = c.strip()
        if len(c) < 4:
            continue
        if c.lower() in STOP_PT:
            continue
        if c not in vistos:
            vistos.add(c)
            ents.append(c)
        if len(ents) >= max_ent:
            break
    return ents

def consulta_externa_sugerida(texto):
    ents = extrair_entidades_simples(texto, max_ent=6)
    kws = extrair_palavras_chave_simples(texto, max_terms=10)
    termos = ents + [k for k in kws if k not in [e.lower() for e in ents]]
    termos = termos[:12]
    return " ".join([f'"{t}"' if " " in t else t for t in termos])

def construir_consulta_contextual(row):
    texto = row.get("text_limpo", row.get("text", row.get("texto_busca_v5", "")))
    lead = texto_curto(texto, 900)
    kws = " ".join(extrair_palavras_chave_simples(texto, max_terms=20))
    ents = " ".join(extrair_entidades_simples(texto, max_ent=12))
    marcadores = str(row.get("marcadores", ""))
    zona = str(row.get("zona_adtc", ""))
    tax_cats = str(row.get("taxonomia_categorias", ""))
    return normalizar_texto(" | ".join([lead, ents, kws, marcadores, zona, tax_cats]))

def construir_base_contextual_expandida():
    partes = []

    # 1. Base curada já existente.
    base_curada = base_rag.copy()
    if "id" not in base_curada.columns:
        base_curada["id"] = [f"curada_{i}" for i in range(len(base_curada))]
    if "tema" not in base_curada.columns:
        base_curada["tema"] = ""
    if "fragmento" not in base_curada.columns:
        base_curada["fragmento"] = ""
    if "fonte_sugerida" not in base_curada.columns:
        base_curada["fonte_sugerida"] = "Base curada ADTC/RAG"
    base_curada["origem_contexto"] = "base_curada_adtc"
    partes.append(base_curada[["id","tema","fragmento","fonte_sugerida","origem_contexto"]].copy())

    # 2. Textos do treino como memória contextual, sem uso de rótulo.
    if "train_pos_trava" in globals():
        treino_ctx = train_pos_trava.copy()
    elif "train_raw" in globals():
        treino_ctx = train_raw.copy()
    else:
        treino_ctx = pd.DataFrame()

    if not treino_ctx.empty:
        col_text = None
        for c in ["text", "texto_modelo", "texto", "text_limpo", "conteudo", "content"]:
            if c in treino_ctx.columns:
                col_text = c
                break
        if col_text is not None:
            tmp = treino_ctx[[col_text]].copy()
            tmp = tmp.reset_index().rename(columns={"index": "id_treino_contexto", col_text: "fragmento"})
            tmp["fragmento"] = tmp["fragmento"].map(lambda x: texto_curto(x, 1800))
            tmp["tema"] = tmp["fragmento"].map(lambda x: " / ".join(extrair_palavras_chave_simples(x, max_terms=8)))
            tmp["id"] = tmp["id_treino_contexto"].map(lambda x: f"idpt_treino_{x}")
            tmp["fonte_sugerida"] = "IDPT News — conjunto de treino; usado apenas como memória textual, sem rótulo"
            tmp["origem_contexto"] = "idpt_treino_sem_rotulo"
            partes.append(tmp[["id","tema","fragmento","fonte_sugerida","origem_contexto"]].copy())

    base_expandida = pd.concat(partes, ignore_index=True)
    base_expandida["fragmento"] = base_expandida["fragmento"].fillna("").astype(str)
    base_expandida = base_expandida[base_expandida["fragmento"].str.len() > 40].drop_duplicates("fragmento").reset_index(drop=True)
    return base_expandida

base_contextual_v7 = construir_base_contextual_expandida()
base_contextual_v7.to_csv(OUT_DIR / "base_rag_contextual_expandida_v7.csv", index=False)

print("Base RAG original:", len(base_rag))
print("Base RAG contextual expandida:", len(base_contextual_v7))
display(base_contextual_v7["origem_contexto"].value_counts().to_frame("n"))

# Cria consultas enriquecidas.
test_contextual = test.copy()
test_contextual["texto_busca_contextual_v7"] = test_contextual.apply(construir_consulta_contextual, axis=1)
test_contextual["consulta_externa_sugerida"] = test_contextual["text"].fillna(test_contextual.get("text_limpo", "")).map(consulta_externa_sugerida)

# Instancia o motor usando a mesma classe, mas com base ampliada e pool maior.
inicio = time.time()
motor_contextual_v7 = MotorRAGHibridoV5(
    base_contextual_v7,
    candidate_pool=max(80, CANDIDATE_POOL),
    peso_lexical=0.30,
    peso_denso=0.70,
)
print("Motor RAG contextual v7:", motor_contextual_v7.motor)
print("Modelo embedding:", motor_contextual_v7.modelo_embedding_usado)
print("Modelo reranker:", motor_contextual_v7.modelo_reranker_usado)
print("Tempo de inicialização:", round(time.time() - inicio, 2), "s")

# Recupera mais candidatos: top 10 para auditoria.
TOP_K_CONTEXTUAL = 10

teste_todo_contextual_v7 = motor_contextual_v7.aplicar(
    test_contextual,
    texto_col="texto_busca_contextual_v7",
    top_k=TOP_K_CONTEXTUAL
)

# Renomeia sufixo v5 para contextual_v7.
rename_ctx = {}
for c in list(teste_todo_contextual_v7.columns):
    if c.endswith("_v5"):
        rename_ctx[c] = c.replace("_v5", "_contextual_v7")
    elif c in ["motor_rag_v5", "modelo_embedding_v5", "modelo_reranker_v5"]:
        rename_ctx[c] = c.replace("_v5", "_contextual_v7")
teste_todo_contextual_v7 = teste_todo_contextual_v7.rename(columns=rename_ctx)

mask_prioritarios = pd.Series(False, index=teste_todo_contextual_v7.index)
if "tipo_erro" in teste_todo_contextual_v7.columns:
    mask_prioritarios = mask_prioritarios | teste_todo_contextual_v7["tipo_erro"].astype(str).isin(["FP","FN"])
if "zona_adtc" in teste_todo_contextual_v7.columns:
    mask_prioritarios = mask_prioritarios | teste_todo_contextual_v7["zona_adtc"].astype(str).str.contains("exige contexto|falso gatilho", case=False, na=False)

rag_prioritarios_contextual_v7 = teste_todo_contextual_v7.loc[mask_prioritarios].copy()

teste_todo_contextual_v7.to_csv(OUT_DIR / "rag_contextual_teste_todo_v7_top10.csv", index=False)
rag_prioritarios_contextual_v7.to_csv(OUT_DIR / "rag_contextual_casos_prioritarios_v7_top10.csv", index=False)

print("RAG contextual — teste todo:", teste_todo_contextual_v7.shape)
print("RAG contextual — casos prioritários:", rag_prioritarios_contextual_v7.shape)


# 17. Planilha final completa para tese

Esta etapa gera uma planilha consolidada com o dataset rotulado e auditado.

A planilha não deve conter apenas a classificação binária. Ela reúne:

- rótulo original;
- predição BERTimbau;
- confiança;
- tipo de acerto/erro;
- zona ADTC;
- marcadores discursivos;
- recursos taxonômicos;
- categorias da taxonomia;
- explicação classificatória;
- RAG contextual top 10;
- consulta externa sugerida;
- XAI local quando houver amostra;
- parecer ADTC-XAI por texto.

Esta é a planilha recomendada para análise qualitativa e para extração de exemplos no capítulo da tese.


In [ ]:

# ============================================================
# 17. Planilha final completa — dataset rotulado + ADTC + Taxonomia + RAG + XAI
# ============================================================

def carregar_csv_opcional(caminhos):
    if isinstance(caminhos, str):
        caminhos = [caminhos]
    for c in caminhos:
        try:
            p = Path(c)
            if p.exists():
                return pd.read_csv(p)
        except Exception:
            pass
    return pd.DataFrame()

def reduzir_texto_planilha(s, n=2500):
    s = str(s) if not pd.isna(s) else ""
    s = re.sub(r"\s+", " ", s).strip()
    return s[:n]

def parecer_adtc_xai(row):
    partes = []
    label = row.get("label", "")
    pred = row.get("bert_pred", "")
    tipo = row.get("tipo_erro", "")
    zona = row.get("zona_adtc", "")
    conf = row.get("bert_conf", "")
    marc = row.get("marcadores", "")
    tax = row.get("taxonomia_recursos", "")
    rag1 = row.get("fragmento_top1_contextual_v7", "")
    fonte1 = row.get("fonte_sugerida_top1_contextual_v7", "")
    lime = row.get("lime_top_tokens", "")
    ig = row.get("ig_top_tokens", "")
    shap = row.get("shap_top_tokens", "")

    partes.append(f"BERTimbau: rótulo={label}; predição={pred}; tipo={tipo}; confiança={conf}.")
    partes.append(f"Zona ADTC: {zona}.")
    if str(marc).strip():
        partes.append(f"Marcadores ADTC: {marc}.")
    else:
        partes.append("Marcadores ADTC: sem marcação superficial relevante registrada.")
    if str(tax).strip():
        partes.append(f"Recursos taxonômicos: {tax}.")
    if str(rag1).strip():
        partes.append(f"Contexto RAG top1: {reduzir_texto_planilha(rag1, 420)} Fonte/observação: {fonte1}.")
    else:
        partes.append("Contexto RAG: nenhum fragmento contextual relevante foi recuperado na posição top1.")
    xai_partes = []
    if str(lime).strip(): xai_partes.append(f"LIME: {lime}")
    if str(ig).strip(): xai_partes.append(f"IG: {ig}")
    if str(shap).strip(): xai_partes.append(f"SHAP: {shap}")
    if xai_partes:
        partes.append("XAI local disponível na amostra: " + " | ".join(xai_partes))
    else:
        partes.append("XAI local: item fora da amostra XAI ou sem tokens agregados.")
    return " ".join(partes)

# Base principal: usa a saída contextual se disponível; senão, triade_v5.
if "teste_todo_contextual_v7" in globals():
    dataset_final = teste_todo_contextual_v7.copy()
elif "triade_v5" in globals():
    dataset_final = triade_v5.copy()
else:
    dataset_final = test.copy()

# Enxuga textos muito longos, mas preserva o texto completo em coluna separada truncada para planilha.
for c in ["text", "text_limpo", "fragmento_top1_contextual_v7", "fragmento_top2_contextual_v7", "fragmento_top3_contextual_v7"]:
    if c in dataset_final.columns:
        dataset_final[c] = dataset_final[c].map(lambda x: reduzir_texto_planilha(x, 3000))

# Integra síntese XAI, quando disponível.
if "sintese_xai" in globals() and isinstance(sintese_xai, pd.DataFrame) and not sintese_xai.empty:
    xai_sint = sintese_xai.copy()
else:
    xai_sint = carregar_csv_opcional([
        OUT_DIR / "xai_bertimbau" / "xai_sintese_adtc_bertimbau.csv",
        "xai_sintese_adtc_bertimbau(2).csv",
        "xai_sintese_adtc_bertimbau.csv",
    ])

if not xai_sint.empty and "id_texto_original" in xai_sint.columns and "id_texto_original" in dataset_final.columns:
    cols_xai = [c for c in [
        "id_texto_original", "lime_top_tokens", "ig_top_tokens", "attention_top_tokens", "shap_top_tokens",
        "xai_model_pred", "xai_model_p_iro", "xai_model_conf", "xai_concorda_predicao_precomputada"
    ] if c in xai_sint.columns]
    dataset_final = dataset_final.merge(
        xai_sint[cols_xai].drop_duplicates("id_texto_original"),
        on="id_texto_original",
        how="left",
        suffixes=("", "_xai")
    )

# Integra avaliação manual, quando disponível.
amostra_manual_final = pd.DataFrame()
if "triade_v5_com_manual" in globals() and isinstance(triade_v5_com_manual, pd.DataFrame):
    amostra_manual_final = triade_v5_com_manual.copy()
elif "amostra_larga_v5" in globals() and isinstance(amostra_larga_v5, pd.DataFrame):
    amostra_manual_final = amostra_larga_v5.copy()

if not amostra_manual_final.empty and "id_texto_original" in amostra_manual_final.columns and "id_texto_original" in dataset_final.columns:
    cols_manual = [c for c in amostra_manual_final.columns if "manual" in c.lower() or c in ["id_texto_original"]]
    dataset_final = dataset_final.merge(
        amostra_manual_final[cols_manual].drop_duplicates("id_texto_original"),
        on="id_texto_original",
        how="left",
        suffixes=("", "_manual")
    )

dataset_final["parecer_adtc_xai"] = dataset_final.apply(parecer_adtc_xai, axis=1)

# Ordena colunas prioritárias para leitura.
cols_prioritarias = [
    "id_texto_original", "text", "text_limpo", "label", "bert_pred", "bert_conf", "bert_p_iro",
    "acertou_bert", "tipo_erro", "tipo_adtc", "zona_adtc", "n_marc_sup", "marcadores",
    "taxonomia_recurso_ids", "taxonomia_recursos", "taxonomia_categorias",
    "explicacao_classificacao_taxonomica", "consulta_externa_sugerida",
    "score_final_top1_contextual_v7", "id_top1_contextual_v7", "tema_top1_contextual_v7",
    "fragmento_top1_contextual_v7", "fonte_sugerida_top1_contextual_v7",
    "score_final_top2_contextual_v7", "tema_top2_contextual_v7", "fragmento_top2_contextual_v7",
    "fonte_sugerida_top2_contextual_v7",
    "score_final_top3_contextual_v7", "tema_top3_contextual_v7", "fragmento_top3_contextual_v7",
    "fonte_sugerida_top3_contextual_v7",
    "lime_top_tokens", "ig_top_tokens", "shap_top_tokens", "attention_top_tokens",
    "parecer_adtc_xai",
]
cols_ordenadas = [c for c in cols_prioritarias if c in dataset_final.columns]
cols_restantes = [c for c in dataset_final.columns if c not in cols_ordenadas]
dataset_final = dataset_final[cols_ordenadas + cols_restantes]

# Abas-resumo.
resumo_contextual = pd.DataFrame({
    "item": [
        "versao_final",
        "dataset",
        "total_itens_teste",
        "bertimbau_retreinado",
        "predicoes_precomputadas",
        "base_rag_original",
        "base_rag_contextual_expandida",
        "rag_contextual_top_k",
        "rag_contextual_casos_prioritarios",
        "taxonomia_recursos",
        "taxonomia_categorias",
        "xai_neural_executada",
        "observacao_metodologica",
    ],
    "valor": [
        "v7_adtc_rag_contextual_xai_final",
        "IDPT News",
        len(dataset_final),
        False,
        True,
        len(base_rag) if "base_rag" in globals() else "NA",
        len(base_contextual_v7) if "base_contextual_v7" in globals() else "NA",
        globals().get("TOP_K_CONTEXTUAL", "NA"),
        len(rag_prioritarios_contextual_v7) if "rag_prioritarios_contextual_v7" in globals() else "NA",
        len(quadro_taxonomia) if "quadro_taxonomia" in globals() and isinstance(quadro_taxonomia, pd.DataFrame) else "NA",
        quadro_taxonomia["categoria_num"].nunique() if "quadro_taxonomia" in globals() and isinstance(quadro_taxonomia, pd.DataFrame) and "categoria_num" in quadro_taxonomia.columns else "NA",
        globals().get("modelo_xai_disponivel", "NA"),
        "O BERTimbau fornece a predição; a ADTC, a taxonomia e o RAG contextual oferecem auditabilidade e explicabilidade discursiva.",
    ]
})

zonas_final = dataset_final["zona_adtc"].value_counts(dropna=False).reset_index() if "zona_adtc" in dataset_final.columns else pd.DataFrame()
if not zonas_final.empty:
    zonas_final.columns = ["zona_adtc", "n"]

tipos_erro_final = dataset_final["tipo_erro"].value_counts(dropna=False).reset_index() if "tipo_erro" in dataset_final.columns else pd.DataFrame()
if not tipos_erro_final.empty:
    tipos_erro_final.columns = ["tipo_erro", "n"]

dicionario_colunas = pd.DataFrame([
    ["label", "Rótulo original do IDPT: 0 = não irônico; 1 = irônico/satírico."],
    ["bert_pred", "Predição pré-computada do BERTimbau clássico."],
    ["bert_conf", "Confiança associada à predição pré-computada."],
    ["tipo_erro", "Tipo de acerto/erro: VP, VN, FP ou FN."],
    ["zona_adtc", "Zona interpretativa da ADTC a partir da predição e dos marcadores."],
    ["marcadores", "Marcadores linguístico-discursivos detectados pela ADTC."],
    ["taxonomia_recursos", "Recursos da Taxonomia dos Recursos Irônicos mapeados no texto."],
    ["taxonomia_categorias", "Categorias taxonômicas mobilizadas."],
    ["consulta_externa_sugerida", "Consulta textual sugerida para pesquisa externa/manual sobre a notícia."],
    ["fragmento_top*_contextual_v7", "Fragmentos recuperados pelo RAG contextual aprimorado."],
    ["parecer_adtc_xai", "Síntese auditável por item: predição, ADTC, taxonomia, RAG e XAI quando disponível."],
], columns=["coluna", "descricao"])

xlsx_final = OUT_DIR / "planilha_final_tese_v7_adtc_rag_contextual_xai_completa.xlsx"

try:
    with pd.ExcelWriter(xlsx_final, engine="openpyxl") as writer:
        resumo_contextual.to_excel(writer, sheet_name="Resumo_metodologico", index=False)
        dataset_final.to_excel(writer, sheet_name="Dataset_rotulado_ADTC_XAI", index=False)
        zonas_final.to_excel(writer, sheet_name="Zonas_ADTC", index=False)
        tipos_erro_final.to_excel(writer, sheet_name="Tipos_erro", index=False)
        if "metricas_bert" in globals():
            metricas_bert.to_excel(writer, sheet_name="Metricas_BERT", index=False)
        if "metricas_rag_v5" in globals():
            metricas_rag_v5.to_excel(writer, sheet_name="Metricas_RAG_manual", index=False)
        if "taxonomia_resumo_categorias" in globals() and isinstance(taxonomia_resumo_categorias, pd.DataFrame):
            taxonomia_resumo_categorias.to_excel(writer, sheet_name="Taxonomia_resumo", index=False)
        if "quadro_taxonomia" in globals() and isinstance(quadro_taxonomia, pd.DataFrame):
            quadro_taxonomia.to_excel(writer, sheet_name="Taxonomia_completa", index=False)
        if "evidencias_taxonomia_textos" in globals() and isinstance(evidencias_taxonomia_textos, pd.DataFrame):
            evidencias_taxonomia_textos.to_excel(writer, sheet_name="Evidencias_taxonomia", index=False)
        if "teste_todo_contextual_v7" in globals():
            teste_todo_contextual_v7.to_excel(writer, sheet_name="RAG_contextual_top10", index=False)
        if "rag_prioritarios_contextual_v7" in globals():
            rag_prioritarios_contextual_v7.to_excel(writer, sheet_name="RAG_prioritarios", index=False)
        if not xai_sint.empty:
            xai_sint.to_excel(writer, sheet_name="XAI_amostra", index=False)
        if not amostra_manual_final.empty:
            amostra_manual_final.to_excel(writer, sheet_name="Amostra_manual_RAG", index=False)
        dicionario_colunas.to_excel(writer, sheet_name="Dicionario_colunas", index=False)

    print("Planilha final completa salva em:", xlsx_final)

except Exception as e:
    warnings.warn(f"Falha ao salvar XLSX final: {repr(e)}. Salvando CSV principal.")
    dataset_final.to_csv(OUT_DIR / "dataset_rotulado_adtc_xai_contextual_final.csv", index=False)

# Relatório técnico curto da correção
relatorio_contextual = f"""
# Relatório técnico — V7 ADTC + RAG contextual + XAI

## Objetivo

Esta versão corrige a limitação da busca contextual observada na avaliação manual do RAG. A predição do BERTimbau não é alterada; o foco é melhorar a capacidade de auditoria, recuperando mais contexto para cada notícia.

## Estratégia

- BERTimbau não retreinado.
- Predições pré-computadas preservadas.
- ADTC mantida como camada principal de explicabilidade discursiva.
- Taxonomia ADTC mantida para nomear recursos irônicos.
- Base RAG expandida de {len(base_rag) if 'base_rag' in globals() else 'NA'} para {len(base_contextual_v7) if 'base_contextual_v7' in globals() else 'NA'} fragmentos/textos.
- Top-k contextual ampliado para {globals().get('TOP_K_CONTEXTUAL', 'NA')}.
- Planilha final completa gerada: `{xlsx_final.name}`.

## Interpretação

A busca contextual deve ser lida como apoio à auditoria, não como classificador. O objetivo é permitir verificar de que trata cada notícia, quais contextos semelhantes são recuperados e se a ironia depende de memória discursiva, falso gatilho ou marcas superficiais.
"""

(OUT_DIR / "RELATORIO_TECNICO_V7_RAG_CONTEXTUAL_XAI.md").write_text(relatorio_contextual, encoding="utf-8")
print("Relatório técnico salvo:", OUT_DIR / "RELATORIO_TECNICO_V7_RAG_CONTEXTUAL_XAI.md")


## Leitura metodológica final

Esta versão foi desenhada para otimizar o papel da ADTC na tese.

A ADTC não substitui o BERTimbau. O BERTimbau permanece como classificador neural de referência. A ADTC acrescenta:

- zonas interpretativas;
- identificação de falsos gatilhos;
- separação entre ironia marcada e ironia dependente de contexto;
- explicitação de marcadores discursivos;
- mapeamento taxonômico dos recursos de ironia;
- organização dos casos prioritários para auditoria;
- planilha final auditável.

O RAG contextual aprimorado deve ser lido como uma camada de apoio: ele ajuda a localizar temas, trechos semelhantes e possíveis contextos de origem, mas não decide sozinho se há ironia.


# 18. RAG integrado ADTC: recuperação semântica + re-ranqueamento discursivo

A versão V7 mostrou que o RAG híbrido retornava candidatos, mas a pertinência contextual global ainda era baixa.  
A melhoria desta versão não consiste em descartar o RAG semântico. Pelo contrário: ele continua sendo útil para ampliar a busca.

A mudança é que o RAG passa a operar em duas fases cooperativas:

1. **Recuperação semântica ampla**  
   O motor híbrido BGE-M3 + reranker recupera candidatos semanticamente próximos.

2. **Re-ranqueamento discursivo orientado pela ADTC**  
   Os candidatos recuperados são reordenados por critérios discursivos:
   - zona ADTC;
   - tipo de erro/acerto;
   - marcadores discursivos;
   - recursos taxonômicos;
   - entidades compartilhadas;
   - tipo de contexto necessário;
   - origem do fragmento;
   - memória discursiva.

Assim, o RAG deixa de ser apenas busca por semelhança e passa a ser uma camada de **busca contextual auditável**.


In [ ]:

# ============================================================
# 18.1 Funções auxiliares — RAG integrado ADTC
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import math
import warnings
import unicodedata

STOP_PT_INTEGRADO = {
    "a","o","as","os","um","uma","uns","umas","de","do","da","dos","das","em","no","na","nos","nas",
    "por","para","com","sem","sob","sobre","entre","até","apos","após","antes","e","ou","mas","que","se",
    "ao","aos","à","às","como","mais","menos","muito","muita","muitos","muitas","também","ja","já","foi",
    "ser","são","era","eram","ter","tem","têm","há","não","sim","sua","seu","suas","seus","ele","ela",
    "eles","elas","isso","isto","aquele","aquela","aqueles","aquelas","este","esta","estes","estas",
    "diz","disse","afirmou","segundo","contra","durante","onde","quando","porque","ainda","apenas",
    "anos","ano","dia","dias","meses","mês","vai","pode","brasil","brasileiro","brasileira"
}

def _norm_int(s):
    s = "" if pd.isna(s) else str(s)
    return re.sub(r"\s+", " ", s).strip()

def _fold_int(s):
    """Minúsculas + remoção de acentos, apenas para comparação de substrings.
    Não altera o texto salvo nas planilhas."""
    s = _norm_int(s).lower()
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))

def _tokens_chave_int(texto, max_terms=24):
    texto = _norm_int(texto)
    toks = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9][A-Za-zÀ-ÖØ-öø-ÿ0-9_-]{3,}", texto)
    freq, ordem = {}, []
    for t in toks:
        tl = t.lower()
        if tl in STOP_PT_INTEGRADO:
            continue
        if tl not in freq:
            ordem.append(tl)
            freq[tl] = 0
        freq[tl] += 1
    return sorted(freq, key=lambda x: (-freq[x], ordem.index(x)))[:max_terms]

def _entidades_int(texto, max_ent=14):
    texto = _norm_int(texto)
    candidatos = re.findall(
        r"\b[A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-Za-zÁÉÍÓÚÂÊÔÃÕÇáéíóúâêôãõç]+(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-Za-zÁÉÍÓÚÂÊÔÃÕÇáéíóúâêôãõç]+){0,3}",
        texto
    )
    vistos, ents = set(), []
    for c in candidatos:
        c = c.strip()
        if len(c) < 4:
            continue
        if c.lower() in STOP_PT_INTEGRADO:
            continue
        if c not in vistos:
            vistos.add(c)
            ents.append(c)
        if len(ents) >= max_ent:
            break
    return ents

def _overlap_jaccard(a, b, mode="tokens"):
    if mode == "entidades":
        sa = set([x.lower() for x in _entidades_int(a, 15)])
        sb = set([x.lower() for x in _entidades_int(b, 15)])
    else:
        sa = set(_tokens_chave_int(a, 30))
        sb = set(_tokens_chave_int(b, 30))
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / max(1, len(sa | sb))

def classificar_contexto_necessario_integrado(row):
    zona = str(row.get("zona_adtc", "")).lower()
    marc = str(row.get("marcadores", "")).lower()
    texto = str(row.get("text", row.get("text_limpo", ""))).lower()
    tipos = []

    if "baixa marcação" in zona or "exige contexto" in zona:
        tipos.append("memoria_discursiva_evento_alvo")
    if "falso gatilho" in zona:
        tipos.append("diferenciar_recurso_jornalistico_de_ironia")
    if any(x in marc for x in ["aspas", "discurso_relatado"]):
        tipos.append("fonte_enunciativa_citacao")
    if any(x in marc for x in ["negacao", "adversativa", "contraste"]):
        tipos.append("contraste_argumentativo")
    if any(x in marc for x in ["avaliativo", "hiperbole", "aumentativo", "diminutivo"]):
        tipos.append("avaliacao_lexical_intensificacao")
    if any(x in texto for x in ["governo","presidente","ministro","congresso","stf","deputado","senador","prefeito"]):
        tipos.append("contexto_politico_institucional")
    if any(x in texto for x in ["covid","pandemia","vacina","cloroquina","gripezinha"]):
        tipos.append("memoria_sanitaria_politica")
    if any(x in texto for x in ["copa","futebol","seleção","gol","7 a 1","neymar"]):
        tipos.append("memoria_esportiva")
    if not tipos:
        tipos.append("contexto_tematico_geral")

    return "; ".join(dict.fromkeys(tipos))

def consulta_integrada_adtc(row):
    texto = _norm_int(row.get("text", row.get("text_limpo", "")))
    lead = texto[:900]
    kws = " ".join(_tokens_chave_int(texto, 22))
    ents = " ".join(_entidades_int(texto, 12))
    zona = _norm_int(row.get("zona_adtc", ""))
    tipo = _norm_int(row.get("tipo_erro", ""))
    marc = _norm_int(row.get("marcadores", ""))
    tax = _norm_int(row.get("taxonomia_recursos", ""))
    cats = _norm_int(row.get("taxonomia_categorias", ""))
    tipo_ctx = classificar_contexto_necessario_integrado(row)

    consulta = " | ".join([
        lead,
        "entidades " + ents,
        "palavras chave " + kws,
        "zona adtc " + zona,
        "tipo erro " + tipo,
        "marcadores " + marc,
        "taxonomia " + tax,
        "categorias " + cats,
        "contexto necessario " + tipo_ctx,
        "noticia origem evento alvo memoria discursiva"
    ])

    if "normalizar_texto" in globals():
        return normalizar_texto(consulta)
    return consulta.lower()

def consulta_externa_integrada(row):
    texto = _norm_int(row.get("text", row.get("text_limpo", "")))
    ents = _entidades_int(texto, 7)
    kws = _tokens_chave_int(texto, 10)
    termos = ents + [k for k in kws if k.lower() not in [e.lower() for e in ents]]
    zona = str(row.get("zona_adtc","")).lower()
    extras = []
    if "baixa marcação" in zona or "exige contexto" in zona:
        extras += ["contexto", "origem", "notícia"]
    if "falso gatilho" in zona:
        extras += ["declaração", "aspas", "notícia real"]
    termos = (termos + extras)[:14]
    return " ".join([f'"{t}"' if " " in t else t for t in termos]).strip()

def prioridade_rag_integrado(row):
    zona = str(row.get("zona_adtc", "")).lower()
    tipo = str(row.get("tipo_erro", "")).upper()
    if "baixa marcação" in zona or "exige contexto" in zona:
        return "alta"
    if "falso gatilho" in zona:
        return "alta"
    if tipo in {"FP", "FN"}:
        return "alta"
    if "ironia predita" in zona:
        return "media"
    return "baixa"

def memoria_discursiva_seed_integrada():
    registros = [
        {
            "id": "memoria_covid_gripezinha",
            "tema": "pandemia covid gripezinha cloroquina vacina negacionismo",
            "fragmento": "Memória discursiva associada à pandemia de Covid-19 no Brasil: termos como gripezinha, cloroquina, isolamento, vacina e negação da gravidade podem funcionar como gatilhos de leitura irônica quando há contraste entre minimização discursiva e gravidade factual.",
            "fonte_sugerida": "Base ADTC de memória discursiva — Covid/pandemia",
            "origem_contexto": "memoria_discursiva_adtc",
            "tipo_contexto": "memoria_sanitaria_politica"
        },
        {
            "id": "memoria_aspas_jornalismo",
            "tema": "aspas discurso relatado jornalismo citação fonte enunciativa",
            "fragmento": "No jornalismo, aspas e discurso relatado podem indicar citação, atribuição de fala, distanciamento enunciativo ou responsabilidade da fonte. Essas marcas podem ser falsos gatilhos para classificadores de ironia, pois nem toda citação ou formulação entre aspas é irônica.",
            "fonte_sugerida": "Base ADTC — aspas e discurso relatado no jornalismo",
            "origem_contexto": "memoria_discursiva_adtc",
            "tipo_contexto": "fonte_enunciativa_citacao"
        },
        {
            "id": "memoria_adversativa_negacao",
            "tema": "negação adversativa concessiva contraste mas porém embora",
            "fragmento": "Negação, adversativas e concessivas sinalizam contraste argumentativo. Esse contraste pode sustentar ironia, mas também é regular em notícia informativa ou texto opinativo. A pertinência depende do alvo, do evento e da posição enunciativa.",
            "fonte_sugerida": "Base ADTC — contraste argumentativo",
            "origem_contexto": "memoria_discursiva_adtc",
            "tipo_contexto": "contraste_argumentativo"
        },
        {
            "id": "memoria_avaliacao_lexical",
            "tema": "adjetivo avaliativo hipérbole aumentativo diminutivo intensificação julgamento",
            "fragmento": "Avaliativos, hipérboles, aumentativos e diminutivos podem intensificar julgamento. Em ironia, produzem tensão entre elogio aparente e crítica implícita; em notícia real, podem apenas relatar avaliação atribuída a uma fonte.",
            "fonte_sugerida": "Base ADTC — avaliação lexical e intensificação",
            "origem_contexto": "memoria_discursiva_adtc",
            "tipo_contexto": "avaliacao_lexical_intensificacao"
        },
        {
            "id": "memoria_7a1",
            "tema": "futebol copa seleção 7 a 1 alemanha vergonha nacional",
            "fragmento": "Memória discursiva do 7 a 1: a derrota da seleção brasileira para a Alemanha em 2014 tornou-se marcador de fracasso, humilhação e comparação irônica em discursos esportivos, políticos e midiáticos.",
            "fonte_sugerida": "Base ADTC de memória discursiva — esporte",
            "origem_contexto": "memoria_discursiva_adtc",
            "tipo_contexto": "memoria_esportiva"
        },
        {
            "id": "memoria_padrao_fifa",
            "tema": "padrão fifa obras públicas estádio superfaturamento promessa legado copa",
            "fragmento": "Memória discursiva de 'padrão FIFA': expressão associada a promessas de qualidade, obras públicas, estádios e críticas a gastos. Pode sustentar ironia por contraste entre promessa institucional e precariedade percebida.",
            "fonte_sugerida": "Base ADTC de memória discursiva — política e Copa",
            "origem_contexto": "memoria_discursiva_adtc",
            "tipo_contexto": "memoria_politica_midiatica"
        },
    ]
    return pd.DataFrame(registros)

def construir_base_rag_integrada():
    partes = []

    # Base curada original
    if "base_rag" in globals() and isinstance(base_rag, pd.DataFrame) and not base_rag.empty:
        base0 = base_rag.copy()
        if "id" not in base0.columns:
            base0["id"] = [f"curada_{i}" for i in range(len(base0))]
        if "tema" not in base0.columns:
            base0["tema"] = ""
        if "fragmento" not in base0.columns:
            base0["fragmento"] = base0.apply(lambda r: " ".join([str(x) for x in r.values]), axis=1)
        if "fonte_sugerida" not in base0.columns:
            base0["fonte_sugerida"] = "Base curada ADTC/RAG"
        if "origem_contexto" not in base0.columns:
            base0["origem_contexto"] = "base_curada_adtc"
        if "tipo_contexto" not in base0.columns:
            base0["tipo_contexto"] = "contexto_curado"
        partes.append(base0[["id","tema","fragmento","fonte_sugerida","origem_contexto","tipo_contexto"]])

    # Memória discursiva explícita
    partes.append(memoria_discursiva_seed_integrada())

    # Treino como memória textual, sem usar rótulos
    treino_ctx = None
    for nome in ["train_pos_trava", "train_raw", "train_df"]:
        if nome in globals() and isinstance(globals()[nome], pd.DataFrame) and not globals()[nome].empty:
            treino_ctx = globals()[nome].copy()
            break

    if treino_ctx is not None:
        col_text = None
        for c in ["text", "texto_modelo", "texto", "text_limpo", "conteudo", "content"]:
            if c in treino_ctx.columns:
                col_text = c
                break
        if col_text is not None:
            tmp = treino_ctx[[col_text]].copy().reset_index().rename(columns={"index": "idx", col_text: "fragmento"})
            tmp["fragmento"] = tmp["fragmento"].map(lambda x: _norm_int(x)[:1800])
            tmp = tmp[tmp["fragmento"].str.len() > 60].drop_duplicates("fragmento").reset_index(drop=True)
            tmp["id"] = tmp.index.map(lambda i: f"idpt_treino_memoria_{i}")
            tmp["tema"] = tmp["fragmento"].map(lambda x: " ".join(_tokens_chave_int(x, 8)))
            tmp["fonte_sugerida"] = "IDPT News — treino usado como memória textual sem rótulo"
            tmp["origem_contexto"] = "idpt_treino_sem_rotulo"
            tmp["tipo_contexto"] = "contexto_tematico_do_corpus"
            partes.append(tmp[["id","tema","fragmento","fonte_sugerida","origem_contexto","tipo_contexto"]])

    base = pd.concat(partes, ignore_index=True)
    base["fragmento"] = base["fragmento"].fillna("").astype(str)
    base = base[base["fragmento"].str.len() > 30].drop_duplicates("fragmento").reset_index(drop=True)
    return base


In [ ]:

# ============================================================
# 18.2 Execução do RAG integrado ADTC
# ============================================================

# DataFrame de teste integrado vindo das etapas anteriores
if "test_contextual" in globals() and isinstance(test_contextual, pd.DataFrame):
    test_integrado_rag = test_contextual.copy()
elif "teste_todo_contextual_v7" in globals() and isinstance(teste_todo_contextual_v7, pd.DataFrame):
    test_integrado_rag = teste_todo_contextual_v7.copy()
elif "triade_v5" in globals() and isinstance(triade_v5, pd.DataFrame):
    test_integrado_rag = triade_v5.copy()
elif "test" in globals() and isinstance(test, pd.DataFrame):
    test_integrado_rag = test.copy()
else:
    raise RuntimeError("Não encontrei DataFrame de teste. Execute as células anteriores da V7.")

test_integrado_rag["tipo_contexto_necessario"] = test_integrado_rag.apply(classificar_contexto_necessario_integrado, axis=1)
test_integrado_rag["consulta_integrada_adtc"] = test_integrado_rag.apply(consulta_integrada_adtc, axis=1)
test_integrado_rag["consulta_externa_sugerida_integrada"] = test_integrado_rag.apply(consulta_externa_integrada, axis=1)
test_integrado_rag["prioridade_rag_integrado"] = test_integrado_rag.apply(prioridade_rag_integrado, axis=1)

base_rag_integrada_adtc = construir_base_rag_integrada()
base_rag_integrada_adtc.to_csv(OUT_DIR / "base_rag_integrada_adtc_v8.csv", index=False)

print("Base RAG original:", len(base_rag) if "base_rag" in globals() else "NA")
print("Base RAG integrada:", len(base_rag_integrada_adtc))
display(base_rag_integrada_adtc["origem_contexto"].value_counts().to_frame("n"))

# Recuperação semântica ampla com o mesmo motor híbrido.
motor_rag_integrado = MotorRAGHibridoV5(
    base_rag_integrada_adtc,
    candidate_pool=max(120, CANDIDATE_POOL if "CANDIDATE_POOL" in globals() else 60),
    peso_lexical=0.30,
    peso_denso=0.70,
)

TOP_K_RAG_INTEGRADO_BRUTO = 20
rag_integrado_bruto = motor_rag_integrado.aplicar(
    test_integrado_rag,
    texto_col="consulta_integrada_adtc",
    top_k=TOP_K_RAG_INTEGRADO_BRUTO
)

# Renomeia as colunas geradas pelo motor.
ren = {}
for c in rag_integrado_bruto.columns:
    if c.endswith("_v5"):
        ren[c] = c.replace("_v5", "_integrado_semantico")
    elif c in ["motor_rag_v5", "modelo_embedding_v5", "modelo_reranker_v5"]:
        ren[c] = c.replace("_v5", "_integrado_semantico")
rag_integrado_bruto = rag_integrado_bruto.rename(columns=ren)

# Correção pós-revisão: o MotorRAGHibridoV5 não emite origem_contexto nem
# tipo_contexto por candidato, então os bônus de origem do score discursivo
# nunca disparavam e a planilha de avaliação manual sairia com origem vazia.
# Preenche por merge de id com a base integrada.
_mapa_origem_int = base_rag_integrada_adtc.set_index("id")["origem_contexto"].astype(str).to_dict()
_mapa_tipo_ctx_int = base_rag_integrada_adtc.set_index("id")["tipo_contexto"].astype(str).to_dict()
for _k in range(1, TOP_K_RAG_INTEGRADO_BRUTO + 1):
    _col_id = f"id_top{_k}_integrado_semantico"
    if _col_id in rag_integrado_bruto.columns:
        rag_integrado_bruto[f"origem_contexto_top{_k}_integrado_semantico"] = (
            rag_integrado_bruto[_col_id].astype(str).map(_mapa_origem_int).fillna("")
        )
        rag_integrado_bruto[f"tipo_contexto_top{_k}_integrado_semantico"] = (
            rag_integrado_bruto[_col_id].astype(str).map(_mapa_tipo_ctx_int).fillna("")
        )

def score_discursivo_candidato(row, k):
    texto = row.get("text", row.get("text_limpo", ""))
    frag = row.get(f"fragmento_top{k}_integrado_semantico", "")
    tema = row.get(f"tema_top{k}_integrado_semantico", "")
    origem = str(row.get(f"origem_contexto_top{k}_integrado_semantico", row.get(f"fonte_sugerida_top{k}_integrado_semantico", "")))
    tipo_ctx = str(row.get("tipo_contexto_necessario", ""))
    marcadores = str(row.get("marcadores", ""))
    zona = str(row.get("zona_adtc", ""))

    if not _norm_int(frag):
        return 0.0

    s_kw = _overlap_jaccard(texto, str(frag) + " " + str(tema), "tokens")
    s_ent = _overlap_jaccard(texto, frag, "entidades")

    # Correção pós-revisão: todas as comparações de substring usam _fold_int
    # (minúsculas sem acento). Antes, tokens sem acento como "memoria",
    # "noticia" e "negacao" não casavam com as formas acentuadas dos
    # fragmentos, e parte dos bônus contextuais ficava sistematicamente muda.
    frag_f = _fold_int(frag)
    tema_f = _fold_int(tema)
    origem_f = _fold_int(origem)
    marcadores_f = _fold_int(marcadores)
    zona_f = _fold_int(zona)
    alvo = origem_f + " " + tema_f + " " + frag_f

    s_ctx = 0.0
    for t in _fold_int(tipo_ctx).split(";"):
        t = t.strip()
        if not t:
            continue
        # compatibilidade explícita com origem/tipo/fragmento
        if t.replace("_", " ") in alvo or any(x in alvo for x in t.split("_")[:2]):
            s_ctx += 0.08

    if "memoria_discursiva_adtc" in origem_f:
        s_ctx += 0.18
    if "idpt_treino" in origem_f:
        s_ctx += 0.04

    # Casos de falso gatilho precisam valorizar fragmentos sobre jornalismo/citação.
    if "falso gatilho" in zona_f and any(x in (frag_f + " " + tema_f) for x in ["aspas", "citacao", "declaracao", "jornalismo"]):
        s_ctx += 0.15

    # Casos de baixa marcação precisam valorizar memória/evento/alvo.
    if ("baixa marcacao" in zona_f or "exige contexto" in zona_f) and any(x in alvo for x in ["memoria", "evento", "alvo", "contexto", "noticia"]):
        s_ctx += 0.15

    # Marcadores e fragmentos com vocabulário compatível
    for m in ["aspas", "negacao", "adversativa", "discurso", "avaliativo", "contraste", "hiperbole"]:
        if m in marcadores_f and m in (frag_f + " " + tema_f):
            s_ctx += 0.04

    return round(min(1.0, 0.38*s_kw + 0.34*s_ent + min(0.28, s_ctx)), 4)

# Re-ranqueia os top-20 por score combinado: score semântico + score discursivo.
def reranquear_linha_integrada(row, top_bruto=20, top_final=10):
    candidatos = []
    for k in range(1, top_bruto + 1):
        frag_col = f"fragmento_top{k}_integrado_semantico"
        if frag_col not in row.index or not _norm_int(row.get(frag_col, "")):
            continue
        score_sem = row.get(f"score_final_top{k}_integrado_semantico", np.nan)
        try:
            score_sem = float(score_sem)
        except Exception:
            score_sem = 0.0
        # Correção pós-revisão: float(np.nan) não levanta exceção, então o
        # except acima não captura score ausente; NaN se propagaria para o
        # score integrado e desestabilizaria a ordenação.
        if not np.isfinite(score_sem):
            score_sem = 0.0
        score_disc = score_discursivo_candidato(row, k)
        score_total = round(0.60*score_sem + 0.40*score_disc, 4)
        candidatos.append({
            "k_original": k,
            "score_semantico": score_sem,
            "score_discursivo": score_disc,
            "score_integrado": score_total,
            "id": row.get(f"id_top{k}_integrado_semantico", ""),
            "tema": row.get(f"tema_top{k}_integrado_semantico", ""),
            "fragmento": row.get(f"fragmento_top{k}_integrado_semantico", ""),
            "fonte_sugerida": row.get(f"fonte_sugerida_top{k}_integrado_semantico", ""),
            "origem_contexto": row.get(f"origem_contexto_top{k}_integrado_semantico", ""),
        })
    candidatos = sorted(candidatos, key=lambda x: (-x["score_integrado"], -x["score_discursivo"], -x["score_semantico"]))
    out = {}
    for i, cand in enumerate(candidatos[:top_final], start=1):
        for key, val in cand.items():
            out[f"{key}_top{i}_rag_integrado"] = val
    out["n_candidatos_rag_integrado"] = len(candidatos)
    out["score_integrado_max_top10"] = candidatos[0]["score_integrado"] if candidatos else 0.0
    out["score_discursivo_max_top10"] = max([c["score_discursivo"] for c in candidatos], default=0.0)
    return pd.Series(out)

rank_integrado = rag_integrado_bruto.apply(reranquear_linha_integrada, axis=1)
rag_integrado_final = pd.concat([rag_integrado_bruto, rank_integrado], axis=1)

rag_integrado_final.to_csv(OUT_DIR / "rag_integrado_adtc_teste_todo_v8_top10.csv", index=False)

rag_integrado_prioritarios = rag_integrado_final[
    rag_integrado_final["prioridade_rag_integrado"].isin(["alta"])
].copy()
rag_integrado_prioritarios.to_csv(OUT_DIR / "rag_integrado_adtc_prioritarios_v8_top10.csv", index=False)

print("RAG integrado final:", rag_integrado_final.shape)
print("Casos prioritários:", rag_integrado_prioritarios.shape)
display(rag_integrado_final["prioridade_rag_integrado"].value_counts().to_frame("n"))


# 19. Avaliação manual do RAG integrado

Esta etapa gera uma planilha específica para avaliação manual.

A avaliação manual continua sendo essencial, porque o notebook não deve declarar sozinho que um fragmento é contexto pertinente.  
Ele deve organizar evidências para que a pesquisadora verifique:

- o trecho ajuda a entender de onde vem a notícia?
- ajuda a identificar tema, evento ou alvo?
- ajuda a diferenciar ironia de falso gatilho?
- ajuda a interpretar a zona ADTC?
- o contexto recuperado é realmente discursivamente pertinente?

A planilha gerada aqui é o instrumento de comprovação empírica da melhoria.


In [ ]:

# ============================================================
# 19. Amostra de avaliação manual — RAG integrado ADTC
# ============================================================

# Amostra: prioriza alta prioridade, mas mantém controles.
alta = rag_integrado_final[rag_integrado_final["prioridade_rag_integrado"].eq("alta")].copy()
media = rag_integrado_final[rag_integrado_final["prioridade_rag_integrado"].eq("media")].copy()
baixa = rag_integrado_final[rag_integrado_final["prioridade_rag_integrado"].eq("baixa")].copy()

n_alta = min(70, len(alta))
n_media = min(20, len(media))
n_baixa = min(10, len(baixa))

amostra_manual_rag_integrado = pd.concat([
    alta.sample(n=n_alta, random_state=42) if n_alta else alta,
    media.sample(n=n_media, random_state=42) if n_media else media,
    baixa.sample(n=n_baixa, random_state=42) if n_baixa else baixa,
], ignore_index=True)

cols_base_manual = [
    "id_texto_original", "text", "label", "bert_pred", "bert_conf", "tipo_erro",
    "zona_adtc", "prioridade_rag_integrado", "marcadores", "taxonomia_recursos",
    "taxonomia_categorias", "tipo_contexto_necessario", "consulta_externa_sugerida_integrada",
]
cols_base_manual = [c for c in cols_base_manual if c in amostra_manual_rag_integrado.columns]

cols_top = []
for k in range(1, 6):
    for c in [
        f"score_integrado_top{k}_rag_integrado",
        f"score_semantico_top{k}_rag_integrado",
        f"score_discursivo_top{k}_rag_integrado",
        f"tema_top{k}_rag_integrado",
        f"fragmento_top{k}_rag_integrado",
        f"fonte_sugerida_top{k}_rag_integrado",
        f"origem_contexto_top{k}_rag_integrado",
    ]:
        if c in amostra_manual_rag_integrado.columns:
            cols_top.append(c)

amostra_manual_rag_integrado = amostra_manual_rag_integrado[cols_base_manual + cols_top].copy()

# Colunas de avaliação manual.
for k in range(1, 6):
    amostra_manual_rag_integrado[f"manual_top{k}_pertinente"] = ""
    amostra_manual_rag_integrado[f"manual_top{k}_ajuda_origem_evento"] = ""
    amostra_manual_rag_integrado[f"manual_top{k}_ajuda_tema"] = ""
    amostra_manual_rag_integrado[f"manual_top{k}_ajuda_ironia_ou_falso_gatilho"] = ""
    amostra_manual_rag_integrado[f"manual_top{k}_observacao"] = ""

amostra_manual_rag_integrado["manual_melhor_topk"] = ""
amostra_manual_rag_integrado["manual_parecer_contextual"] = ""
amostra_manual_rag_integrado["manual_decisao_sobre_contexto"] = ""

arquivo_manual_integrado = OUT_DIR / "amostra_avaliacao_manual_rag_integrado_adtc_v8_para_preencher.xlsx"
amostra_manual_rag_integrado.to_excel(arquivo_manual_integrado, index=False)

# Métricas automáticas descritivas, sem substituir o manual.
diagnostico_rag_integrado = pd.DataFrame({
    "item": [
        "total_teste",
        "total_prioridade_alta",
        "total_prioridade_media",
        "total_prioridade_baixa",
        "base_rag_integrada",
        "amostra_manual_gerada",
        "score_integrado_top1_medio",
        "score_discursivo_top1_medio",
        "score_integrado_top10_max_medio",
    ],
    "valor": [
        len(rag_integrado_final),
        int((rag_integrado_final["prioridade_rag_integrado"] == "alta").sum()),
        int((rag_integrado_final["prioridade_rag_integrado"] == "media").sum()),
        int((rag_integrado_final["prioridade_rag_integrado"] == "baixa").sum()),
        len(base_rag_integrada_adtc),
        len(amostra_manual_rag_integrado),
        float(rag_integrado_final["score_integrado_top1_rag_integrado"].mean()) if "score_integrado_top1_rag_integrado" in rag_integrado_final.columns else np.nan,
        float(rag_integrado_final["score_discursivo_top1_rag_integrado"].mean()) if "score_discursivo_top1_rag_integrado" in rag_integrado_final.columns else np.nan,
        float(rag_integrado_final["score_integrado_max_top10"].mean()) if "score_integrado_max_top10" in rag_integrado_final.columns else np.nan,
    ]
})
diagnostico_rag_integrado.to_csv(OUT_DIR / "diagnostico_rag_integrado_adtc_v8.csv", index=False)

print("Amostra manual salva em:", arquivo_manual_integrado)
display(diagnostico_rag_integrado)


# 20. Planilha final completa para tese

Esta etapa gera a planilha final do notebook melhorado.

A aba principal reúne:

- dataset rotulado;
- predição BERTimbau;
- confiança;
- tipo de erro/acerto;
- zona ADTC;
- marcadores;
- taxonomia;
- tipo de contexto necessário;
- RAG integrado top 10;
- consulta externa sugerida;
- XAI exploratória, quando disponível;
- parecer final auditável.

Essa é a planilha recomendada para extrair exemplos para a tese.


In [ ]:

# ============================================================
# 20. Planilha final V8 — BERTimbau + ADTC + Taxonomia + RAG integrado + XAI
# ============================================================

def _carregar_xai_sintese():
    candidatos = [
        OUT_DIR / "xai_bertimbau" / "xai_sintese_adtc_bertimbau.csv",
        Path("xai_sintese_adtc_bertimbau(2).csv"),
        Path("xai_sintese_adtc_bertimbau.csv"),
    ]
    for p in candidatos:
        if p.exists():
            try:
                return pd.read_csv(p)
            except Exception:
                pass
    return pd.DataFrame()

dataset_final_v8 = rag_integrado_final.copy()
xai_sintese_integrada = _carregar_xai_sintese()

if not xai_sintese_integrada.empty and "id_texto_original" in xai_sintese_integrada.columns and "id_texto_original" in dataset_final_v8.columns:
    cols_xai = [c for c in [
        "id_texto_original", "lime_top_tokens", "ig_top_tokens", "attention_top_tokens",
        "shap_top_tokens", "xai_model_pred", "xai_model_p_iro", "xai_model_conf",
        "xai_concorda_predicao_precomputada"
    ] if c in xai_sintese_integrada.columns]
    dataset_final_v8 = dataset_final_v8.merge(
        xai_sintese_integrada[cols_xai].drop_duplicates("id_texto_original"),
        on="id_texto_original",
        how="left"
    )

def parecer_final_integrado(row):
    partes = []
    partes.append(f"BERTimbau: rótulo={row.get('label','')}; predição={row.get('bert_pred','')}; tipo={row.get('tipo_erro','')}; confiança={row.get('bert_conf','')}.")
    partes.append(f"ADTC: zona={row.get('zona_adtc','')}; prioridade RAG={row.get('prioridade_rag_integrado','')}.")
    partes.append(f"Contexto necessário: {row.get('tipo_contexto_necessario','')}.")
    if _norm_int(row.get("marcadores", "")):
        partes.append(f"Marcadores: {row.get('marcadores','')}.")
    if _norm_int(row.get("taxonomia_recursos", "")):
        partes.append(f"Taxonomia: {row.get('taxonomia_recursos','')}.")
    frag1 = row.get("fragmento_top1_rag_integrado", "")
    if _norm_int(frag1):
        frag1 = _norm_int(frag1)[:520]
        partes.append(
            f"RAG integrado top1: {frag1} "
            f"(score integrado={row.get('score_integrado_top1_rag_integrado','')}; "
            f"score discursivo={row.get('score_discursivo_top1_rag_integrado','')})."
        )
    else:
        partes.append("RAG integrado: sem fragmento top1 recuperado.")
    xai = []
    for c, nome in [("lime_top_tokens","LIME"),("ig_top_tokens","IG"),("shap_top_tokens","SHAP")]:
        if c in row and _norm_int(row.get(c, "")):
            xai.append(f"{nome}: {row.get(c)}")
    if xai:
        partes.append("XAI exploratória: " + " | ".join(xai))
    return " ".join(partes)

dataset_final_v8["parecer_final_adtc_rag_integrado_xai"] = dataset_final_v8.apply(parecer_final_integrado, axis=1)

# Reduz textos para Excel.
for c in dataset_final_v8.columns:
    if dataset_final_v8[c].dtype == "object" and any(x in c.lower() for x in ["text", "fragmento", "consulta", "parecer"]):
        dataset_final_v8[c] = dataset_final_v8[c].map(lambda x: _norm_int(x)[:3200])

prior_cols = [
    "id_texto_original", "text", "label", "bert_pred", "bert_conf", "bert_p_iro",
    "acertou_bert", "tipo_erro", "zona_adtc", "prioridade_rag_integrado",
    "n_marc_sup", "marcadores", "taxonomia_recurso_ids", "taxonomia_recursos",
    "taxonomia_categorias", "tipo_contexto_necessario",
    "consulta_externa_sugerida_integrada", "consulta_integrada_adtc",
    "score_integrado_top1_rag_integrado", "score_semantico_top1_rag_integrado",
    "score_discursivo_top1_rag_integrado", "tema_top1_rag_integrado",
    "fragmento_top1_rag_integrado", "fonte_sugerida_top1_rag_integrado",
    "origem_contexto_top1_rag_integrado",
    "score_integrado_top2_rag_integrado", "score_discursivo_top2_rag_integrado",
    "tema_top2_rag_integrado", "fragmento_top2_rag_integrado", "fonte_sugerida_top2_rag_integrado",
    "score_integrado_top3_rag_integrado", "score_discursivo_top3_rag_integrado",
    "tema_top3_rag_integrado", "fragmento_top3_rag_integrado", "fonte_sugerida_top3_rag_integrado",
    "score_integrado_max_top10", "score_discursivo_max_top10",
    "lime_top_tokens", "ig_top_tokens", "shap_top_tokens", "attention_top_tokens",
    "parecer_final_adtc_rag_integrado_xai",
]
ordered = [c for c in prior_cols if c in dataset_final_v8.columns] + [c for c in dataset_final_v8.columns if c not in prior_cols]
dataset_final_v8 = dataset_final_v8[ordered]

resumo_v8_integrado = pd.DataFrame([
    ["versao", "v8_rag_integrado_adtc_xai_final"],
    ["objetivo", "Melhorar a V7 com RAG semântico e re-ranqueamento discursivo trabalhando juntos."],
    ["bertimbau_retreinado", False],
    ["predicoes_precomputadas", True],
    ["total_teste", len(dataset_final_v8)],
    ["base_rag_integrada", len(base_rag_integrada_adtc)],
    ["casos_prioridade_alta", int((rag_integrado_final["prioridade_rag_integrado"] == "alta").sum())],
    ["amostra_manual_rag_integrado", len(amostra_manual_rag_integrado)],
    ["arquivo_amostra_manual", "amostra_avaliacao_manual_rag_integrado_adtc_v8_para_preencher.xlsx"],
    ["arquivo_planilha_final", "planilha_final_tese_v8_rag_integrado_adtc_xai.xlsx"],
], columns=["item", "valor"])

zonas_v8 = dataset_final_v8["zona_adtc"].value_counts(dropna=False).reset_index() if "zona_adtc" in dataset_final_v8.columns else pd.DataFrame()
if not zonas_v8.empty:
    zonas_v8.columns = ["zona_adtc", "n"]

tipos_v8 = dataset_final_v8["tipo_erro"].value_counts(dropna=False).reset_index() if "tipo_erro" in dataset_final_v8.columns else pd.DataFrame()
if not tipos_v8.empty:
    tipos_v8.columns = ["tipo_erro", "n"]

dicionario_v8 = pd.DataFrame([
    ["consulta_integrada_adtc", "Consulta enriquecida por texto, entidades, palavras-chave, zona ADTC, tipo de erro, marcadores e taxonomia."],
    ["tipo_contexto_necessario", "Tipo de contexto que a ADTC indica como necessário para auditar o caso."],
    ["prioridade_rag_integrado", "Prioridade de acionamento do RAG: alta, média ou baixa."],
    ["score_semantico_top*_rag_integrado", "Score do estágio de recuperação semântica."],
    ["score_discursivo_top*_rag_integrado", "Score de re-ranqueamento discursivo ADTC."],
    ["score_integrado_top*_rag_integrado", "Combinação de score semântico e discursivo."],
    ["consulta_externa_sugerida_integrada", "Consulta sugerida para busca externa/manual da origem e do tema da notícia."],
    ["parecer_final_adtc_rag_integrado_xai", "Síntese auditável por item integrando BERTimbau, ADTC, taxonomia, RAG integrado e XAI exploratória."],
], columns=["coluna", "descricao"])

xlsx_v8_integrado = OUT_DIR / "planilha_final_tese_v8_rag_integrado_adtc_xai.xlsx"

with pd.ExcelWriter(xlsx_v8_integrado, engine="openpyxl") as writer:
    resumo_v8_integrado.to_excel(writer, sheet_name="Resumo", index=False)
    dataset_final_v8.to_excel(writer, sheet_name="Dataset_ADTC_RAG_XAI", index=False)
    zonas_v8.to_excel(writer, sheet_name="Zonas_ADTC", index=False)
    tipos_v8.to_excel(writer, sheet_name="Tipos_erro", index=False)
    diagnostico_rag_integrado.to_excel(writer, sheet_name="Diagnostico_RAG_integrado", index=False)
    base_rag_integrada_adtc.to_excel(writer, sheet_name="Base_RAG_integrada", index=False)
    amostra_manual_rag_integrado.to_excel(writer, sheet_name="Amostra_manual_preencher", index=False)
    if "metricas_bert" in globals():
        metricas_bert.to_excel(writer, sheet_name="Metricas_BERT", index=False)
    if "metricas_rag_v5" in globals():
        metricas_rag_v5.to_excel(writer, sheet_name="Metricas_RAG_anterior", index=False)
    if "quadro_taxonomia" in globals() and isinstance(quadro_taxonomia, pd.DataFrame):
        quadro_taxonomia.to_excel(writer, sheet_name="Taxonomia_ADTC", index=False)
    if not xai_sintese_integrada.empty:
        xai_sintese_integrada.to_excel(writer, sheet_name="XAI_sintese", index=False)
    dicionario_v8.to_excel(writer, sheet_name="Dicionario_colunas", index=False)

print("Planilha final salva em:", xlsx_v8_integrado)


# 21. Relatório técnico V8 integrado

Esta etapa gera um relatório técnico em Markdown, explicando por que esta versão melhora a V7.

O relatório não apresenta dois RAGs em disputa. Ele registra um único fluxo integrado:

> recuperação semântica ampla + re-ranqueamento discursivo ADTC + avaliação manual.


In [ ]:

# ============================================================
# 21. Relatório técnico V8 integrado
# ============================================================

relatorio_integrado = f"""# Relatório técnico — V8 RAG integrado ADTC + XAI

## Objetivo

Esta versão melhora a V7 sem colocar RAG semântico e RAG discursivo em competição. O objetivo é fazê-los trabalhar em conjunto.

O BERTimbau não é retreinado. As predições pré-computadas são preservadas. A ADTC atua como camada de auditoria e como orientadora do re-ranqueamento contextual.

## Arquitetura

1. BERTimbau pré-computado produz a decisão inicial.
2. ADTC identifica zonas interpretativas e marcadores.
3. Taxonomia ADTC nomeia os recursos de ironia.
4. O RAG híbrido recupera candidatos semanticamente próximos.
5. A ADTC re-ranqueia os candidatos por pertinência discursiva.
6. A avaliação manual verifica se o contexto recuperado realmente ajuda.
7. A XAI exploratória adiciona inspeção local quando disponível.

## Diferença em relação à V7

A V7 recuperava contexto com RAG híbrido, mas a avaliação manual mostrou baixa precisão contextual global.  
A V8 mantém o motor semântico, mas adiciona orientação discursiva:

- tipo de contexto necessário;
- prioridade por zona ADTC;
- consulta enriquecida;
- memória discursiva seed;
- base expandida com treino usado como memória textual sem rótulo;
- score discursivo;
- score integrado.

## Dados da execução

- Total do teste: {len(rag_integrado_final)}
- Base RAG integrada: {len(base_rag_integrada_adtc)}
- Casos de prioridade alta para RAG: {int((rag_integrado_final["prioridade_rag_integrado"] == "alta").sum())}
- Amostra manual gerada: {len(amostra_manual_rag_integrado)}

## Arquivos gerados

- `base_rag_integrada_adtc_v8.csv`
- `rag_integrado_adtc_teste_todo_v8_top10.csv`
- `rag_integrado_adtc_prioritarios_v8_top10.csv`
- `diagnostico_rag_integrado_adtc_v8.csv`
- `amostra_avaliacao_manual_rag_integrado_adtc_v8_para_preencher.xlsx`
- `planilha_final_tese_v8_rag_integrado_adtc_xai.xlsx`

## Interpretação para a tese

A melhoria do RAG não decorre apenas de trocar o modelo de embedding. Ela decorre de integrar recuperação semântica com critérios discursivos. Assim, o RAG não é tratado como substituto das condições de produção, mas como mecanismo de busca de evidências contextuais orientado pela ADTC e validado manualmente.

## Formulação recomendada

A V8 transforma o RAG em um componente integrado à ADTC: primeiro recupera candidatos por similaridade semântica; depois os reordena segundo pertinência discursiva, zona interpretativa, marcadores e taxonomia. Com isso, a busca contextual deixa de ser uma lista de textos parecidos e passa a compor uma trilha auditável para análise da ironia.
"""

(OUT_DIR / "RELATORIO_TECNICO_V8_RAG_INTEGRADO_ADTC_XAI.md").write_text(relatorio_integrado, encoding="utf-8")
print("Relatório técnico salvo em:", OUT_DIR / "RELATORIO_TECNICO_V8_RAG_INTEGRADO_ADTC_XAI.md")
